# DataInsight | Time series: Forecasting hourly traffic volume on Interstate 94

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trthminh/CO5177/blob/main/notebooks/timeseries.ipynb)

| | |
|---|---|
| **Course** | Programming Foundations for Data Analysis and Visualization (CO5177), Semester 261, 2026–2027 |
| **Lecturer** | Lê Thành Sách, Faculty of Computer Science and Engineering, HCMUT – VNU-HCM |
| **Group** | DataInsight: Trương Thanh Minh (2580893), Tô Anh Phát (2580894), Trần Lê Hoàng Long (2570443) |
| **Owner of this notebook** | Trương Thanh Minh (2580893) |
| **Dataset** | [Metro Interstate Traffic Volume](https://www.kaggle.com/datasets/pooriamst/metro-interstate-traffic-volume), MnDOT via the UCI Machine Learning Repository, CC BY 4.0 |
| **Task** | Forecast `traffic_volume` 1 hour ahead from its history, the calendar (hour, weekday, holidays) and the weather |

**Problem statement.** The Minnesota Department of Transportation counts every vehicle that passes station ATR 301 on the westbound I-94 between Minneapolis and St Paul. We forecast the number of vehicles in the **next hour** from the past counts, the calendar and the weather observed one hour earlier. A reliable one-hour forecast supports ramp metering, incident response and travel-time information. Models are compared on a chronological validation period with **MAE** (vehicles/hour, the deciding metric), RMSE, MAPE and skill over naive baselines; the chosen model is scored once on the test period.

**Table of contents**
1. [Setup](#s1)
2. [Data loading](#s2)
3. [Dataset description](#s3)
4. [Data quality checks](#s4)
5. [Exploratory data analysis (EDA)](#s5)
6. [Data preparation](#s6)
7. [Modeling](#s7)
8. [Evaluation on the test set](#s8)
9. [Inference on new data](#s9)
10. [Conclusion](#s10)

> **How to run.** Open in Google Colab → **Runtime → Run all**. The notebook installs missing libraries and downloads the data by itself; no manual upload, Drive mount or API key is needed. Run all takes about 5 minutes (most of it in sections 7.1–7.2).

<a id="s1"></a>
## 1. Setup

### 1.1 Install missing libraries

Google Colab already ships most of these libraries. The cell below only installs a package when Python cannot find it, so Run all stays fast on Colab and still works on a fresh local environment. `statsmodels` ≥ 0.14 is needed for `MSTL` (several seasonal cycles at once).

In [1]:
import importlib.util
import subprocess
import sys

# import name -> pip requirement. Add a line here for every extra library you use.
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn>=1.2",
    "plotly": "plotly>=5.15",
    "gdown": "gdown>=4.6",
    "kagglehub": "kagglehub",
    "statsmodels": "statsmodels>=0.14",
}

# find_spec returns None when a module is not installed; nothing is imported yet, so no restart is needed.
missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

All required packages are already installed.


### 1.2 Imports

In [2]:
import itertools
import time
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from statsmodels.tsa.seasonal import MSTL, STL           # seasonal-trend decomposition (several / one cycle)
from statsmodels.tsa.statespace.sarimax import SARIMAX   # regression with ARIMA errors (section 7.1)
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf  # autocorrelation and stationarity tests

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge                   # linear regression with L2 penalty: stable with correlated lags
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit  # CV folds that always train on the past
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", module="statsmodels")  # convergence chatter of SARIMAX
warnings.filterwarnings("ignore", category=InterpolationWarning)  # KPSS reports p-values clipped to [0.01, 0.10]
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

numpy 2.2.6 | pandas 2.2.3 | scikit-learn 1.5.2


### 1.3 Configuration

**Everything that depends on the chosen dataset is set here and only here.** Periods, lags and windows are counted in **steps of `FREQ`** (hourly data: 24 steps = 1 day, 168 steps = 1 week). The section that justifies each value is given in the comment.

In [3]:
# ---- Reproducibility --------------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)

# ---- Data source (section 2) ------------------------------------------------------------------
DATA_DIR = Path("data")
DATA_SOURCE = {"kind": "url",
               "location": "https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz",
               "filename": "Metro_Interstate_Traffic_Volume.csv.gz"}
READ_KWARGS = {"keep_default_na": False}   # keep the string "None" of `holiday` instead of turning it into NaN

# ---- Series definition ------------------------------------------------------------------------
TIME_COL = "date_time"
TARGET = "traffic_volume"        # vehicles per hour, westbound
FREQ = "h"
NUM_EXOG = ["temp_c", "rain_1h", "snow_1h", "clouds_all", "n_weather"]   # temp_c and n_weather are built in 2.2
CAT_EXOG = ["weather_main", "weather_description"]
DUPLICATE_AGG = "mean"           # duplicated hours repeat the same traffic count (4.1)
WINDOW_START, WINDOW_END = "2016-01-01 00:00", "2018-09-30 23:00"   # 4.2: before 2016 the series has gaps of up to 10 months
MAJOR_HOLIDAYS = ["New Years Day", "Memorial Day", "Independence Day", "Labor Day",
                  "Thanksgiving Day", "Christmas Day"]                  # 5.8: traffic drops by a third or more

# ---- Forecasting setup ------------------------------------------------------------------------
HORIZON = 1                      # forecast this many steps ahead; only information >= HORIZON steps old is used
SEASONAL_PERIODS = [24, 168]     # 5.5: daily and weekly cycles
LAGS = [1, 2, 3, 24, 168]        # 5.6: short-term memory + same hour yesterday + same hour last week
ROLL_WINDOWS = [24]              # rolling mean/std over the past day
MAX_INTERP_GAP = 12              # 4.2: the longest gap inside the window is 9 hours, so every gap there is filled
TRAIN_END, VAL_END = "2017-11-30 23:00", "2018-04-30 23:00"   # test = May-Sep 2018
VALID_RANGES = {"temp_c": (-60, 60), "rain_1h": (0, 100)}      # 4.4: 0 K and 9,831 mm/h are sensor errors
EVAL_ON_OBSERVED_ONLY = True     # score models only on time steps that were observed, not interpolated
FOURIER_K_GRID = [21, 42, 63]    # 7.1: number of weekly harmonics tried for the SARIMAX model
ARIMA_ORDER = (3, 0, 0)          # 5.6: PACF cuts off after a few lags, no differencing needed (d = 0)

# Stop Run all early with a clear message instead of failing 20 cells later.
for _name, _value in {"DATA_SOURCE": DATA_SOURCE, "TARGET": TARGET, "FREQ": FREQ, "TRAIN_END": TRAIN_END, "VAL_END": VAL_END}.items():
    assert "TODO" not in str(_value), f"Fill in {_name} in section 1.3 before running the notebook."

LAGS = sorted(k for k in LAGS if k >= HORIZON)  # a lag shorter than the horizon is not known at forecast time
print("Lags used:", LAGS)

Lags used: [1, 2, 3, 24, 168]


### 1.4 Plot style and shared helpers

All figures use **Plotly** (interactive: hover shows exact values, zoom works on long series). One shared template keeps colours and fonts identical across the three notebooks of the group. `show_fig(fig, title)` numbers the figures automatically, and every title states the **conclusion** of the figure.

Distribution charts are **pre-computed with NumPy** (histograms, quantiles) instead of sending tens of thousands of points to Plotly, which keeps the notebook small and fast to open.

In [4]:
# Colours by role, fixed order: categorical series take BLUE, ORANGE, AQUA, YELLOW, then RED/MUTED.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
RED, MUTED, GRID = "#e34948", "#898781", "#e1e0d9"
INK_2 = "#52514e"                                                                            # secondary text
QUALITATIVE = [BLUE, ORANGE, AQUA, YELLOW, RED, MUTED]
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # counts, intensities
DIVERGING = [[0, BLUE], [0.5, "#f0efec"], [1, RED]]                                          # correlations (-1..1)

# A named Plotly template = one place to change the look of every chart.
pio.templates["datainsight"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    colorway=QUALITATIVE,
    paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    title=dict(x=0.01, font=dict(size=16)),
    xaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color=INK_2),
    yaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color=INK_2),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
    margin=dict(l=60, r=20, t=80, b=50),
))
pio.templates.default = "plotly_white+datainsight"
# Store figures as Plotly JSON (interactive in Colab / Jupyter) instead of embedding plotly.js in every output.
pio.renderers.default = "colab" if "google.colab" in sys.modules else "plotly_mimetype"

DAY_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

_figure_number = itertools.count(1)  # global counter: Run all numbers figures 1, 2, 3, ... in order


def show_fig(fig, title, height=420, **layout):
    """Give the figure its number and conclusion-style title, then display it."""
    fig.update_layout(title=f"Figure {next(_figure_number)}. {title}", height=height, **layout)
    fig.show()


def hist_bar(values, bins=40, color=BLUE, name=None):
    """Histogram pre-computed with NumPy: the notebook stores ~40 bars instead of every data point."""
    values = np.asarray(values, dtype=float)
    counts, edges = np.histogram(values[~np.isnan(values)], bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.92, marker_color=color, name=name,
                  hovertemplate="%{x:,.2f}: %{y:,}<extra></extra>")


def box_from_groups(series, groups, color=BLUE, name=None):
    """Box plot from pre-computed quantiles per group (box = 25-75 %, whiskers = 5-95 %)."""
    q = series.groupby(groups).quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack()
    return go.Box(x=q.index, q1=q[0.25], median=q[0.5], q3=q[0.75], lowerfence=q[0.05], upperfence=q[0.95],
                  marker_color=color, line_width=1.5, name=name)


def band(fig, x, lo, hi, color, name, row=None, col=None):
    """Filled band between two lines lo and hi."""
    kw = dict(row=row, col=col) if row else {}
    fig.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[hi, lo[::-1]], fill="toself", fillcolor=color,
                             line=dict(width=0), hoverinfo="skip", name=name, showlegend=False), **kw)


def grid_position(i, ncols):
    """(row, col) in a make_subplots grid for the i-th panel, filled left to right."""
    return i // ncols + 1, i % ncols + 1

<a id="s2"></a>
## 2. Data loading

### 2.1 Download

The data is **never uploaded by hand**: the notebook fetches it from a public source and caches it in `data/` (ignored by git), so the lecturer only has to press Run all. `download_dataset` supports three kinds of source:

| `kind` | `location` | Library | Notes |
|---|---|---|---|
| `"url"` | direct download link | `urllib` (standard library) | GitHub raw, UCI, Hugging Face `resolve/...` links, … |
| `"gdrive"` | Google Drive **file id** (the part after `/d/` in the share link) | `gdown` | Share the file as *Anyone with the link → Viewer* |
| `"kaggle"` | dataset handle `owner/dataset-name` | `kagglehub` | Public datasets need no API key |

This dataset uses `"url"`: the gzip-compressed CSV of the UCI Machine Learning Repository (the Kaggle page mirrors the same file).

In [5]:
def download_dataset(source, data_dir=DATA_DIR):
    """Download one data source (skipped if already cached) and return the path of `source["filename"]`."""
    data_dir.mkdir(parents=True, exist_ok=True)
    kind, location, filename = source["kind"], source["location"], source["filename"]

    if kind == "kaggle":
        import kagglehub
        root = Path(kagglehub.dataset_download(location))  # kagglehub keeps its own cache
    else:
        root = data_dir
        archive = source.get("archive")
        target = data_dir / (archive or filename)
        if not target.exists():  # cache: never download twice
            if kind == "url":
                urllib.request.urlretrieve(location, target)
            elif kind == "gdrive":
                import gdown
                gdown.download(id=location, output=str(target), quiet=False)
            else:
                raise ValueError(f"Unknown source kind {kind!r}: use 'url', 'gdrive' or 'kaggle'.")
        if archive:
            with zipfile.ZipFile(target) as zf:
                zf.extractall(data_dir)

    matches = sorted(root.rglob(filename))  # rglob: the file may sit in a sub-folder of the archive
    if not matches:
        found = [p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()][:20]
        raise FileNotFoundError(f"{filename!r} not found under {root}. Files found: {found}")
    print(f"Data file: {matches[0]} ({matches[0].stat().st_size / 1e6:,.1f} MB)")
    return matches[0]


def read_table(path, na_values=None, **kwargs):
    """Read a data file with the pandas reader that matches its extension (.gz: pandas decompresses it)."""
    suffix = (Path(path.stem).suffix if path.suffix.lower() == ".gz" else path.suffix).lower()
    if suffix in {".csv", ".txt", ".data"}:
        return pd.read_csv(path, na_values=na_values, **kwargs)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t", na_values=na_values, **kwargs)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, na_values=na_values, **kwargs)
    if suffix == ".parquet":
        return pd.read_parquet(path, **kwargs)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl", **kwargs)
    raise ValueError(f"No reader configured for {suffix!r}; add one in read_table.")

### 2.2 Read the table and build the timestamp

Two columns are derived right after reading, because they are row-wise unit changes, not statistics:
- `temp_c`: the temperature in °C instead of Kelvin (easier to read; `VALID_RANGES` is written in °C);
- `n_weather`: how many weather rows the source recorded for the same hour. Section 4.1 shows why this count is needed.

In [6]:
def build_timestamp(df):
    """Return one datetime per row. Unparseable values become NaT and are counted in section 4.1."""
    return pd.to_datetime(df[TIME_COL], errors="coerce")


data_path = download_dataset(DATA_SOURCE)
raw = read_table(data_path, **READ_KWARGS)
raw.columns = raw.columns.str.strip()
raw["timestamp"] = build_timestamp(raw)
assert TARGET in raw.columns, f"TARGET {TARGET!r} not found. Columns: {list(raw.columns)}"

raw["temp_c"] = raw["temp"] - 273.15
raw["n_weather"] = raw.groupby("timestamp")["weather_main"].transform("size")

summary = pd.Series({
    "rows": f"{len(raw):,}",
    "first timestamp": raw["timestamp"].min(),
    "last timestamp": raw["timestamp"].max(),
    "time span": raw["timestamp"].max() - raw["timestamp"].min(),
    "expected steps at FREQ": f"{len(pd.date_range(raw['timestamp'].min(), raw['timestamp'].max(), freq=FREQ)):,}",
}, name="value")
display(summary.to_frame())
raw.head()

Data file: data\Metro_Interstate_Traffic_Volume.csv.gz (0.4 MB)


,value
rows,"48,204"
first timestamp,2012-10-02 09:00:00
last timestamp,2018-09-30 23:00:00
time span,2189 days 14:00:00
expected steps at FREQ,"52,551"


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,timestamp,temp_c,n_weather
0,None,288.280,0.000,0.000,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,2012-10-02 09:00:00,15.130,1
1,None,289.360,0.000,0.000,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,2012-10-02 10:00:00,16.210,1
2,None,289.580,0.000,0.000,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,2012-10-02 11:00:00,16.430,1
3,None,290.130,0.000,0.000,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,2012-10-02 12:00:00,16.980,1
4,None,291.140,0.000,0.000,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,2012-10-02 13:00:00,17.990,1


<a id="s3"></a>
## 3. Dataset description

The Minnesota Department of Transportation (MnDOT) counts vehicles at the automatic traffic recorder **ATR 301**, roughly midway between Minneapolis and St Paul, on the **westbound** lanes of I-94. The source joined each hourly count with the weather of that hour (OpenWeatherMap) and US / Minnesota holidays. One row = one hour (in theory, see 4.1), local time (US Central), October 2012 to September 2018.

| Column | Type | Meaning | Unit |
|---|---|---|---|
| `date_time` | datetime | start of the hour | local time (CST/CDT) |
| `traffic_volume` | integer | **value to forecast**: vehicles in that hour | vehicles/hour |
| `holiday` | categorical | holiday name, `None` otherwise | |
| `temp` | float | mean temperature (converted to `temp_c`) | K (°C) |
| `rain_1h`, `snow_1h` | float | rain / snow in the hour | mm |
| `clouds_all` | integer | cloud cover | % |
| `weather_main` | categorical | weather group (Clear, Clouds, Rain, Snow, …) | |
| `weather_description` | categorical | finer description (light rain, heavy snow, …) | |

In [7]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              48204 non-null  object        
 1   temp                 48204 non-null  float64       
 2   rain_1h              48204 non-null  float64       
 3   snow_1h              48204 non-null  float64       
 4   clouds_all           48204 non-null  int64         
 5   weather_main         48204 non-null  object        
 6   weather_description  48204 non-null  object        
 7   date_time            48204 non-null  object        
 8   traffic_volume       48204 non-null  int64         
 9   timestamp            48204 non-null  datetime64[ns]
 10  temp_c               48204 non-null  float64       
 11  n_weather            48204 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(3), object(4)
memory usage: 4.4+ MB


In [8]:
raw[[TARGET, *NUM_EXOG]].describe().T  # min/max reveal impossible values (e.g. 0 K temperature)

,count,mean,std,min,25%,50%,75%,max
traffic_volume,"48,204.000","3,259.818","1,986.861",0.000,"1,193.000","3,380.000","4,933.000","7,280.000"
temp_c,"48,204.000",8.056,13.338,-273.150,-0.990,9.300,18.656,36.920
rain_1h,"48,204.000",0.334,44.789,0.000,0.000,0.000,0.000,"9,831.300"
snow_1h,"48,204.000",0.000,0.008,0.000,0.000,0.000,0.000,0.510
clouds_all,"48,204.000",49.362,39.016,0.000,1.000,64.000,90.000,100.000
n_weather,"48,204.000",1.424,0.788,1.000,1.000,1.000,2.000,6.000


In [9]:
cat_cols = ["holiday", *CAT_EXOG]
pd.DataFrame({"distinct values": raw[cat_cols].nunique(), "most frequent": raw[cat_cols].mode().iloc[0]})

,distinct values,most frequent
holiday,12,None
weather_main,11,Clouds
weather_description,38,sky is clear


In [10]:
wd = raw.groupby(["weather_main", "weather_description"]).size().rename("n").reset_index()
wm = wd.groupby("weather_main")["n"].sum().sort_values(ascending=False)
palette = QUALITATIVE[:5] + ["#7b5fd6", "#3fa7c9", "#b4782f", MUTED, "#d65f9e", "#5e8f3a"]
color_of = dict(zip(wm.index, palette))

fig = go.Figure(go.Sunburst(
    ids=list(wm.index) + [f"{m}/{d}" for m, d in zip(wd["weather_main"], wd["weather_description"])],
    labels=list(wm.index) + list(wd["weather_description"]),
    parents=[""] * len(wm) + list(wd["weather_main"]),
    values=list(wm.values) + list(wd["n"]),
    branchvalues="total",
    marker=dict(colors=[color_of[m] for m in wm.index] + [color_of[m] for m in wd["weather_main"]],
                line=dict(color="#ffffff", width=1.5)),
    insidetextorientation="radial",
    hovertemplate="%{label}: %{value:,} rows (%{percentRoot:.1%})<extra></extra>",
))
show_fig(fig, f"Clouds and Clear cover {(wm.iloc[:2].sum() / wm.sum()):.0%} of rows; "
              f"{len(wm)} weather groups split into {len(wd)} descriptions",
         height=620, margin=dict(t=70, l=10, r=10, b=10))

**Observation.** 48,204 rows cover October 2012 to September 2018, but the calendar would hold 52,551 hours, so rows are both missing and duplicated (section 4). The target ranges from 0 to 7,280 vehicles/hour with a mean of about 3,260. Two values already look impossible in the summary: `temp_c` reaches −273 °C (0 K) and `rain_1h` reaches 9,831 mm. The 11 weather groups split into 38 descriptions; `Clouds` and `Clear` dominate while `Squall` or `Smoke` appear only a few times. Only 61 rows carry a holiday name (section 4.6 explains why so few).

<a id="s4"></a>
## 4. Data quality checks

A regular time series needs **exactly one row per time step and no missing step**: lags (`shift`) and rolling windows count rows, not time. This section diagnoses; fixes are applied in section 6.1.

### 4.1 Timestamps: parse errors, order, duplicates

In [11]:
ts_valid = raw["timestamp"].notna()
dup_mask = raw["timestamp"].duplicated(keep=False) & ts_valid
groups_differing = raw[dup_mask].groupby("timestamp")[TARGET].nunique().gt(1)

timestamp_checks = pd.DataFrame({
    "check": ["unparseable timestamps (NaT)", "rows already in time order",
              "extra rows sharing a timestamp", f"duplicated timestamps whose {TARGET} differs"],
    "result": [int((~ts_valid).sum()), bool(raw.loc[ts_valid, "timestamp"].is_monotonic_increasing),
               int(raw.loc[ts_valid, "timestamp"].duplicated().sum()), int(groups_differing.sum())],
})
display(timestamp_checks)
print("Most common steps between consecutive timestamps:")
raw.loc[ts_valid, "timestamp"].drop_duplicates().sort_values().diff().value_counts().head(5)

,check,result
0,unparseable timestamps (NaT),0
1,rows already in time order,True
2,extra rows sharing a timestamp,7629
3,duplicated timestamps whose traffic_volume dif...,0


Most common steps between consecutive timestamps:


timestamp
0 days 01:00:00    37986
0 days 02:00:00     2192
0 days 03:00:00      201
0 days 04:00:00       59
0 days 05:00:00       33
Name: count, dtype: int64

Within a group of rows sharing one hour, which columns actually differ?

In [12]:
n_distinct = raw[dup_mask].groupby("timestamp")[["holiday", TARGET, "temp", "rain_1h", "snow_1h", "clouds_all", *CAT_EXOG]].nunique()
differs = (n_distinct > 1).mean().mul(100).sort_values()

fig = go.Figure(go.Bar(x=differs.values, y=differs.index, orientation="h",
                       marker_color=np.where(differs.values > 50, ORANGE, MUTED),
                       text=[f"{v:.1f}%" for v in differs.values], textposition="outside", textfont_color=INK_2,
                       hovertemplate="%{y}: %{x:.1f}% of duplicate groups differ<extra></extra>"))
show_fig(fig, "Within a duplicated hour only the weather columns differ; the traffic count is repeated",
         height=380, showlegend=False, xaxis_range=[0, 112],
         xaxis_title="% of duplicate groups where the column has more than one value")

In [13]:
example_time = raw.loc[dup_mask, "timestamp"].value_counts().idxmax()
raw.loc[raw["timestamp"] == example_time, ["date_time", TARGET, "temp", "weather_main", "weather_description", "n_weather"]]

,date_time,traffic_volume,temp,weather_main,weather_description,n_weather
6162,2013-05-19 10:00:00,3591,287.150,Thunderstorm,thunderstorm with light rain,6
6163,2013-05-19 10:00:00,3591,287.150,Mist,mist,6
6164,2013-05-19 10:00:00,3591,287.150,Thunderstorm,proximity thunderstorm,6
6165,2013-05-19 10:00:00,3591,287.150,Rain,moderate rain,6
6166,2013-05-19 10:00:00,3591,287.150,Rain,light rain,6
6167,2013-05-19 10:00:00,3591,287.150,Thunderstorm,thunderstorm with heavy rain,6


**Observation.** Timestamps all parse, are in order and are spaced by 1 hour, but **7,629 rows (15.8%) are extra rows** for an hour that already exists. In those groups the traffic count never differs; only `weather_main` / `weather_description` change (in 98% of the groups). When several weather phenomena happen in the same hour (light rain *and* mist *and* a thunderstorm in the example), the source writes one row per phenomenon and **copies the traffic count**. Merging them with `DUPLICATE_AGG = "mean"` is therefore exact for the target; the categorical columns keep the first phenomenon, and `n_weather` keeps the number of phenomena so the information is not lost.

### 4.2 Missing time steps

`to_regular` merges duplicates and then `asfreq(FREQ)` builds the **complete calendar**: every expected time step gets a row, and steps that were never recorded appear as `NaN`. A *gap* is a run of consecutive missing steps.

In [14]:
def to_regular(df):
    """One row per timestamp (duplicates merged), reindexed on the full calendar at FREQ."""
    agg = {TARGET: DUPLICATE_AGG, **{c: DUPLICATE_AGG for c in NUM_EXOG}, **{c: "first" for c in CAT_EXOG}}
    out = df.dropna(subset=["timestamp"]).groupby("timestamp").agg(agg).sort_index()
    return out.asfreq(FREQ)


def gap_table(missing):
    """Start, end and length of every run of consecutive missing steps, longest first."""
    run_id = missing.ne(missing.shift()).cumsum()[missing]
    steps = pd.Series(missing.index[missing], index=run_id.values)
    return (steps.groupby(level=0).agg(start="min", end="max", steps="size")
                 .sort_values("steps", ascending=False).reset_index(drop=True))


regular = to_regular(raw)
is_missing = regular[TARGET].isna()
gaps = gap_table(is_missing)
print(f"{len(regular):,} expected steps | {is_missing.sum():,} missing ({is_missing.mean():.1%}) | {len(gaps):,} gaps")
gaps.head(6)

52,551 expected steps | 11,976 missing (22.8%) | 2,588 gaps


,start,end,steps
0,2014-08-08 02:00:00,2015-06-11 19:00:00,7386
1,2013-10-27 02:00:00,2013-11-06 03:00:00,242
2,2015-06-14 21:00:00,2015-06-19 17:00:00,117
3,2014-04-29 09:00:00,2014-05-04 04:00:00,116
4,2015-10-23 12:00:00,2015-10-27 07:00:00,92
5,2014-07-27 17:00:00,2014-07-30 08:00:00,64


In [15]:
daily_raw = regular[TARGET].resample("D").mean()
big = gaps.iloc[0]

fig = go.Figure(go.Scatter(x=daily_raw.index, y=daily_raw.values, mode="lines", line=dict(color=BLUE, width=1.3),
                           name="daily mean", hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f} vehicles/hour<extra></extra>"))
fig.add_vrect(x0=big.start, x1=big.end, fillcolor=ORANGE, opacity=0.12, line_width=0,
              annotation_text=f"{big.steps:,} consecutive hours missing", annotation_position="top left",
              annotation_font_color=INK_2)
fig.add_vrect(x0=pd.Timestamp(WINDOW_START), x1=pd.Timestamp(WINDOW_END), fillcolor=BLUE, opacity=0.06, line_width=0,
              annotation_text="study window", annotation_position="top right", annotation_font_color=BLUE)
show_fig(fig, f"Daily mean traffic 2012–2018: a {big.steps / 24:.0f}-day hole in 2014–2015 and many breaks before 2016",
         yaxis_title="Vehicles/hour (daily mean)", showlegend=False)

In [16]:
COVERAGE_FREQ = "MS"  # month start: one cell per month in the heatmap
coverage = (~is_missing).resample(COVERAGE_FREQ).mean() * 100
cov_grid = (coverage.to_frame("pct").assign(year=coverage.index.year, month=coverage.index.month)
                    .pivot(index="year", columns="month", values="pct").reindex(columns=range(1, 13)))

fig = go.Figure(go.Heatmap(z=cov_grid.values, x=MONTH_LABELS, y=cov_grid.index.astype(str),
                           colorscale=[[0, RED], [0.5, "#f6d7c3"], [0.9, "#cde2fb"], [1, BLUE]], zmin=0, zmax=100,
                           text=[["" if np.isnan(v) else f"{v:.0f}" for v in row] for row in cov_grid.values],
                           texttemplate="%{text}", textfont=dict(size=11), xgap=3, ygap=3,
                           colorbar=dict(title="% hours<br>observed"),
                           hovertemplate="%{x} %{y}: %{z:.1f}% of hours observed<extra></extra>"))
show_fig(fig, "Share of hours observed per month: 2014–2015 is mostly empty, 2016–2018 is almost complete",
         height=380, yaxis_autorange="reversed")

In [17]:
bins = [0, 1, 3, MAX_INTERP_GAP, 168, np.inf]
labels = ["1 hour", "2–3 hours", f"4–{MAX_INTERP_GAP} hours", f"{MAX_INTERP_GAP + 1} hours – 7 days", "> 7 days"]
gap_bucket = pd.cut(gaps["steps"], bins=bins, labels=labels)
n_gaps = gap_bucket.value_counts().reindex(labels)
hours_lost = gaps.groupby(gap_bucket, observed=False)["steps"].sum().reindex(labels)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("Number of gaps", "Hours lost (share of all missing hours)"))
fig.add_trace(go.Bar(x=labels, y=n_gaps.values, marker_color=BLUE, text=n_gaps.values, textposition="outside",
                     textfont_color=INK_2, hovertemplate="%{x}: %{y:,} gaps<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=hours_lost.values, marker_color=ORANGE,
                     text=[f"{v / hours_lost.sum():.0%}" for v in hours_lost.values], textposition="outside",
                     textfont_color=INK_2, hovertemplate="%{x}: %{y:,} hours<extra></extra>"), row=1, col=2)
fig.update_yaxes(range=[0, n_gaps.max() * 1.18], row=1, col=1)
fig.update_yaxes(range=[0, hours_lost.max() * 1.18], row=1, col=2)
show_fig(fig, f"{n_gaps.iloc[0] / n_gaps.sum():.0%} of gaps last 1 hour, but {hours_lost.iloc[-1] / hours_lost.sum():.0%} "
              "of missing hours come from gaps longer than a week", height=420, showlegend=False)

In [18]:
in_window = is_missing[WINDOW_START:WINDOW_END]
pd.DataFrame({
    "period": ["before the window (10/2012 – 12/2015)", "study window (01/2016 – 09/2018)"],
    "% hours observed": [100 * (~is_missing[:WINDOW_START].iloc[:-1]).mean(), 100 * (~in_window).mean()],
    "missing hours": [int(is_missing[:WINDOW_START].iloc[:-1].sum()), int(in_window.sum())],
    "longest gap (hours)": [gap_table(is_missing[:WINDOW_START].iloc[:-1])["steps"].max(), gap_table(in_window)["steps"].max()],
}).set_index("period")

,% hours observed,missing hours,longest gap (hours)
period,,,
before the window (10/2012 – 12/2015),61.469,10964,7386
study window (01/2016 – 09/2018),95.800,1012,9


**Observation.** 11,976 of 52,551 hours (22.8%) are missing, in 2,588 gaps. Most gaps last a single hour (the counter briefly drops out) and are safe to interpolate, but most *missing hours* come from a few gaps longer than a week, above all **7,386 consecutive hours (August 2014 to June 2015)**. No interpolation can rebuild ten months, and the weekly lag `lag_168` would be wrong around such holes.

**Decision.** Keep the study window **01/2016 – 09/2018** (`WINDOW_START`, `WINDOW_END`): 2 years and 9 months, enough to see every season at least twice. Inside it 95.8% of the hours are observed and the longest gap is **9 hours**, so `MAX_INTERP_GAP = 12` lets section 6.1 fill every gap by time interpolation (each filled hour is flagged `is_imputed`).

### 4.3 Missing values per column (after reindexing)

In [19]:
na = regular.loc[WINDOW_START:WINDOW_END].isna().sum()
pd.DataFrame({"missing steps in the window": na, "% of steps": na / len(regular.loc[WINDOW_START:WINDOW_END]) * 100})

,missing steps in the window,% of steps
traffic_volume,1012,4.200
temp_c,1012,4.200
rain_1h,1012,4.200
snow_1h,1012,4.200
clouds_all,1012,4.200
n_weather,1012,4.200
weather_main,1012,4.200
weather_description,1012,4.200


**Observation.** Inside the window every column is missing on exactly the same 1,012 hours (4.2%): the raw file has no empty cells, only absent hours. The same rule (interpolate numbers, forward-fill categories) therefore fixes every column.

### 4.4 Impossible values

Physically impossible values (negative counts, 0 K temperature, rainfall above world records) are **measurement errors**: they become `NaN` in 6.1 and are then treated like missing steps.

In [20]:
rows = []
for c, (low, high) in VALID_RANGES.items():
    too_low = int((regular[c] < low).sum()) if low is not None else 0
    too_high = int((regular[c] > high).sum()) if high is not None else 0
    rows.append({"column": c, "rule": f"[{low}, {high}]", "below": too_low, "above": too_high,
                 "observed min": regular[c].min(), "observed max": regular[c].max()})
pd.DataFrame(rows) if rows else print("VALID_RANGES is empty: add physical limits in section 1.3.")

,column,rule,below,above,observed min,observed max
0,temp_c,"[-60, 60]",10,0,-273.150,36.920
1,rain_1h,"[0, 100]",0,1,0.000,"9,831.300"


In [21]:
rain_pos = raw.loc[raw["rain_1h"] > 0, "rain_1h"].to_numpy()
fig = make_subplots(rows=1, cols=2, subplot_titles=("Temperature (°C)", "Hourly rainfall when > 0 (log10 mm)"))
fig.add_trace(hist_bar(raw["temp_c"], bins=80, color=BLUE), row=1, col=1)
fig.add_trace(hist_bar(np.log10(rain_pos), bins=60, color=AQUA), row=1, col=2)
fig.add_annotation(x=-273, y=1, text="0 K", showarrow=True, arrowcolor=RED, ay=-40, font_color=RED, row=1, col=1)
fig.add_annotation(x=np.log10(rain_pos.max()), y=0, text=f"{rain_pos.max():,.0f} mm", showarrow=True, arrowcolor=RED,
                   ay=-40, font_color=RED, row=1, col=2)
fig.update_xaxes(title_text="°C", row=1, col=1)
fig.update_xaxes(title_text="log10(mm)", row=1, col=2)
fig.update_yaxes(title_text="Hours (log scale)", type="log", dtick=1)
show_fig(fig, "Two weather columns contain sensor errors far outside the physical range", height=400, showlegend=False)

**Observation.** 10 hours (two winter nights in 2014) report exactly 0 K, and one hour in July 2016 reports 9,831 mm of rain, about 30 times the world record for one hour; the next largest value is 56 mm. These are **sensor errors**, not weather, so `VALID_RANGES` turns them into `NaN` and section 6.1 interpolates them like missing hours. Extreme values of the *target* are a different matter (4.5).

### 4.5 Outliers of the target: robust z-score

A global z-score fails on seasonal data (night values look like outliers next to daytime values). So:
1. remove the **seasonal profile**: the median of all steps at the same position in the longest cycle (same hour of the week) → residual $r_t$;
2. compute a **robust** z-score in a rolling window of one cycle: $z_t = \dfrac{r_t - \text{median}(r)}{1.4826 \cdot \text{MAD}(r)}$. Median and MAD are not pulled by the outliers themselves (a rolling mean/std would be, see question 5.9); the constant 1.4826 puts the MAD on the scale of a standard deviation;
3. flag $|z_t| > 3.5$ (Iglewicz–Hoaglin threshold).

The check runs on the study window. Flagged values are **kept** if they are real events (holiday, storm, closure) and only used to analyse errors in section 8.5.

In [22]:
Z_THRESHOLD = 3.5
P = max(SEASONAL_PERIODS)
y_reg = regular.loc[WINDOW_START:WINDOW_END, TARGET]
position = pd.Series(np.arange(len(y_reg)) % P, index=y_reg.index)     # position inside the longest cycle
profile = y_reg.groupby(position).transform("median")                  # usual value at that position
resid = y_reg - profile
window = resid.rolling(P, center=True, min_periods=P // 2)
mad = window.apply(lambda a: np.nanmedian(np.abs(a - np.nanmedian(a))), raw=True)
z_robust = (resid - window.median()) / (1.4826 * mad)
is_outlier = z_robust.abs() > Z_THRESHOLD

print(f"Outliers: {int(is_outlier.sum()):,} steps ({is_outlier.mean():.2%})")
fig = go.Figure()
fig.add_trace(go.Scattergl(x=y_reg.index, y=y_reg.values, mode="lines", line=dict(color=BLUE, width=1), name=TARGET))
fig.add_trace(go.Scattergl(x=y_reg.index[is_outlier.to_numpy()], y=y_reg[is_outlier], mode="markers",
                           name="robust |z| > 3.5", marker=dict(color=RED, size=5)))
show_fig(fig, f"{int(is_outlier.sum()):,} hours ({is_outlier.mean():.1%}) deviate strongly from the usual value "
              "for that hour of the week", height=420, yaxis_title="Vehicles/hour",
         xaxis=dict(rangeslider=dict(visible=True, thickness=0.06)))

Outliers: 1,016 steps (4.22%)


**Observation.** About 4–5% of the hours are flagged. Zooming in shows that they come in **blocks of consecutive hours** rather than isolated points: whole days with far less traffic than usual. Question 5.9 identifies them (holidays, snowstorms, a road closure in July 2016) and shows why a non-robust detector misses most of them. **Decision:** they are real events, so the values are kept; `is_outlier` is only used to analyse errors in 8.5.

### 4.6 Holiday labels only on the 00:00 row

In [23]:
holiday_rows = raw[raw["holiday"] != "None"]
print("Hour of day of the rows that carry a holiday name:", holiday_rows["timestamp"].dt.hour.value_counts().to_dict())
holiday_rows[["date_time", "holiday", TARGET]].head(6)

Hour of day of the rows that carry a holiday name: {0: 61}


,date_time,holiday,traffic_volume
126,2012-10-08 00:00:00,Columbus Day,455
1123,2012-11-12 00:00:00,Veterans Day,1000
1370,2012-11-22 00:00:00,Thanksgiving Day,919
2360,2012-12-25 00:00:00,Christmas Day,803
2559,2013-01-01 00:00:00,New Years Day,1439
3697,2013-02-18 00:00:00,Washingtons Birthday,556


**Observation.** All 61 holiday names sit on the **00:00 row**; the other 23 hours of the same day say `None`. Used as is, a model would learn that a holiday lasts one hour. The fix is a domain rule: **the label of the 00:00 row applies to the whole day**. `holiday_of` implements it once and is reused by the EDA (5.8) and the features (6.2). It only looks at the calendar, which is known in advance, so it does not leak future information.

In [24]:
holiday_by_day = holiday_rows.set_index(holiday_rows["timestamp"].dt.normalize())["holiday"]
holiday_by_day = holiday_by_day[~holiday_by_day.index.duplicated()]


def holiday_of(index):
    """Holiday name ("None" if none) for every timestamp of `index`, spread over the whole day."""
    return pd.Series(index.normalize().map(holiday_by_day), index=index).fillna("None")


print(f"{len(holiday_by_day)} holiday days in the data, "
      f"{holiday_by_day[WINDOW_START:WINDOW_END].size} of them in the study window")

53 holiday days in the data, 28 of them in the study window


<a id="s5"></a>
## 5. Exploratory data analysis (EDA)

EDA runs on the study window of `regular` (complete calendar, the 1,012 missing hours left as `NaN`, so they show as breaks). Daily totals, decomposition and ACF need a gap-free series and use a temporary interpolated copy `y_filled`. Every subsection asks a question, answers it with a figure and states the consequence for modeling.

In [25]:
eda = regular.loc[WINDOW_START:WINDOW_END]
y_filled = y_reg.interpolate(method="time", limit_direction="both")   # temporary gap-free copy, EDA only
hol_name = holiday_of(y_reg.index)
day_total = y_filled.resample("D").sum()
is_hol_day = hol_name.resample("D").first().ne("None")
print(f"{len(y_reg):,} hours, {int(y_reg.isna().sum()):,} missing, {int(is_hol_day.sum())} holiday days")

24,096 hours, 1,012 missing, 28 holiday days


### 5.1 Overview: level and long-term trend

In [26]:
OVERVIEW_FREQ = "D"
overview = y_reg.resample(OVERVIEW_FREQ).mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=overview.index, y=overview.values, mode="lines", line=dict(color=BLUE, width=1),
                         opacity=0.75, name="daily mean", hovertemplate="%{x|%a %d/%m/%Y}: %{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=overview.index, y=overview.rolling(28, center=True, min_periods=10).mean().values,
                         mode="lines", line=dict(color=ORANGE, width=3), name="28-day rolling mean",
                         hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f}<extra></extra>"))
show_fig(fig, "Daily mean traffic 2016–2018: no long-term upward or downward trend, dips around new year",
         height=430, yaxis_title="Vehicles/hour (daily mean)", hovermode="x unified",
         xaxis=dict(rangeslider=dict(visible=True, thickness=0.06)))

The line shows only daily means. To see **all 24,096 hours at once**, each day becomes a column and each hour a row, a "fingerprint" of the series:

In [27]:
fp = y_reg.to_frame("v").assign(date=y_reg.index.normalize(), hour=y_reg.index.hour).pivot(index="hour", columns="date", values="v")
fig = go.Figure(go.Heatmap(z=fp.values, x=fp.columns, y=fp.index, colorscale=SEQ_BLUE, zmin=0, zmax=7000,
                           colorbar=dict(title="Vehicles<br>/hour"),
                           hovertemplate="%{x|%a %d/%m/%Y} %{y}h: %{z:,.0f}<extra></extra>"))
show_fig(fig, "Every hour of 2016–2018: weekday rush hours, pale weekend stripes, holidays and closures as white lines",
         height=460, yaxis=dict(title="Hour of day", dtick=3, autorange="reversed"))

**How to read it.** Each pixel is one hour. The dark horizontal bands around 6–8h and 15–17h are the **morning and afternoon rush hours**; they are cut at regular intervals by **pale vertical stripes**, the Saturdays and Sundays (weekly cycle). Isolated white lines in the middle of a week are holidays, snowstorms or the road closure of July 2016. No region gets steadily darker or lighter over time, so there is **no long-term trend**.

The calendar below sums each day and arranges the days by week × weekday:

In [28]:
cal_df = day_total.to_frame("total").assign(year=day_total.index.year, dow=day_total.index.dayofweek)
jan1_dow = pd.to_datetime(cal_df["year"].astype(str) + "-01-01").dt.dayofweek.to_numpy()
cal_df["week"] = (day_total.index.dayofyear - 1 + jan1_dow) // 7
years = sorted(cal_df["year"].unique())

fig = make_subplots(rows=len(years), cols=1, vertical_spacing=0.06, subplot_titles=[str(yr) for yr in years])
for i, yr in enumerate(years, start=1):
    part = cal_df[cal_df["year"] == yr]
    grid = part.pivot(index="dow", columns="week", values="total").reindex(index=range(7))
    dates = part.assign(d=part.index.strftime("%a %d/%m/%Y")).pivot(index="dow", columns="week", values="d").reindex(index=range(7))
    fig.add_trace(go.Heatmap(z=grid.values, x=grid.columns, y=DAY_LABELS, customdata=dates.values,
                             colorscale=SEQ_BLUE, zmin=day_total.quantile(0.01), zmax=day_total.quantile(0.99),
                             xgap=2, ygap=2, showscale=(i == 1), colorbar=dict(title="Vehicles<br>/day", len=0.9),
                             hovertemplate="%{customdata}: %{z:,.0f} vehicles<extra></extra>"), row=i, col=1)
    fig.update_yaxes(autorange="reversed", row=i, col=1)
    fig.update_xaxes(range=[-0.5, 53.5], showgrid=False, row=i, col=1)
fig.update_xaxes(title_text="Week of year", row=len(years), col=1)
show_fig(fig, "Calendar of daily totals: weekends are always pale, holidays stand out as pale weekdays", height=560)

**Observation.** The two bottom rows (Sat, Sun) are always paler: the weekly cycle is very regular. The isolated pale cells on weekdays fall on holidays (Memorial Day at the end of May, 4 July, Thanksgiving at the end of November, Christmas – New Year) and on a few winter storm days. The daily mean stays around 3,300 vehicles/hour across the three years, with short dips around the new year.

### 5.2 A close-up: what does one cycle look like?

In [29]:
ZOOM_STEPS = 2 * max(SEASONAL_PERIODS)  # two of the longest cycles
zoom = y_reg.loc["2017-10-02":"2017-10-15"]
fig = go.Figure(go.Scatter(x=zoom.index, y=zoom.values, mode="lines", line=dict(color=BLUE, width=2),
                           fill="tozeroy", fillcolor="rgba(42,120,214,0.10)",
                           hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f} vehicles<extra></extra>"))
for d in pd.date_range("2017-10-07", "2017-10-15", freq="7D"):
    fig.add_vrect(x0=d, x1=d + pd.Timedelta(days=2), fillcolor=MUTED, opacity=0.12, line_width=0,
                  annotation_text="weekend", annotation_position="top left", annotation_font_color=INK_2)
show_fig(fig, "Two weeks of October 2017: two peaks on weekdays, one lower and flatter peak on weekends",
         height=400, yaxis_title="Vehicles/hour", showlegend=False)

**Observation.** Every weekday has the same shape: almost empty at night (< 500 vehicles), a sharp morning peak around 7h (≈ 6,500), a plateau around 4,500–5,000 at midday and an afternoon peak around 16h. Weekends rise slowly to one broad midday peak of ≈ 4,500–5,000. The series is driven by **two nested cycles**, one day and one week.

### 5.3 Distribution of the target

In [30]:
fig = go.Figure(hist_bar(y_reg, bins=np.arange(0, 7600, 200)))
fig.add_vrect(x0=0, x1=1000, fillcolor=MUTED, opacity=0.08, line_width=0, annotation_text="night",
              annotation_position="top left", annotation_font_color=INK_2)
fig.add_vrect(x0=4000, x1=7400, fillcolor=BLUE, opacity=0.06, line_width=0, annotation_text="daytime",
              annotation_position="top right", annotation_font_color=BLUE)
fig.add_vline(x=y_reg.median(), line_dash="dot", line_color=INK_2, annotation_text=f"median {y_reg.median():,.0f}",
              annotation_position="top left", annotation_font_color=INK_2)
show_fig(fig, f"{TARGET} is bimodal: night hours below 1,000 and daytime hours between 4,000 and 6,000",
         xaxis_title="Vehicles/hour", yaxis_title="Hours", showlegend=False)

In [31]:
v = y_reg.dropna()
print(f"share of hours < 1,000: {(v < 1000).mean():.1%} | 1,000–4,000: {((v >= 1000) & (v < 4000)).mean():.1%} "
      f"| >= 4,000: {(v >= 4000).mean():.1%}")
y_reg.describe().to_frame()

share of hours < 1,000: 21.9% | 1,000–4,000: 33.4% | >= 4,000: 44.7%


,traffic_volume
count,"23,084.000"
mean,"3,299.578"
std,"1,968.709"
min,0.000
25%,"1,269.750"
50%,"3,481.500"
75%,"4,929.000"
max,"7,280.000"


The two humps are really **two groups of hours**. Splitting the distribution by hour of day (dark band = middle 50% of the hours, light band = 90%, line = median):

In [32]:
is_wkend = np.where(y_reg.index.dayofweek >= 5, "Weekend", "Weekday (Mon–Fri)")
qs = y_reg.groupby([is_wkend, y_reg.index.hour]).quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack()
hours = np.arange(24)

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=("Weekday (Mon–Fri)", "Weekend"))
for c, (name, color, rgb) in enumerate([("Weekday (Mon–Fri)", BLUE, "42,120,214"), ("Weekend", ORANGE, "235,104,52")], start=1):
    q = qs.loc[name]
    band(fig, hours, q[0.05].values, q[0.95].values, f"rgba({rgb},0.15)", "5–95%", row=1, col=c)
    band(fig, hours, q[0.25].values, q[0.75].values, f"rgba({rgb},0.35)", "25–75%", row=1, col=c)
    fig.add_trace(go.Scatter(x=hours, y=q[0.5].values, mode="lines+markers", line=dict(color=color, width=2.5),
                             marker=dict(size=6), name="median", showlegend=False,
                             hovertemplate="%{x}h: median %{y:,.0f}<extra>" + name + "</extra>"), row=1, col=c)
fig.update_xaxes(title_text="Hour of day", dtick=3)
fig.update_yaxes(title_text="Vehicles/hour", col=1)
show_fig(fig, "Within one hour of the day the spread is narrow: the hour explains most of the variation",
         height=440)

**Observation.** The distribution is **bimodal, not normal**: about 22% of the hours carry fewer than 1,000 vehicles (night) and 45% carry 4,000 or more (daytime). The standard deviation (≈ 1,970) is about 60% of the mean (≈ 3,300). But for a **given hour** the middle 50% band is narrow, so most of the variation is explained by the clock, not by noise. Consequences: a model must know the hour (calendar features or `lag_24` / `lag_168`); MAPE will be inflated by the night hours where small absolute errors are large relative errors, which is why MAE is the deciding metric. The 5–95% band of weekdays dips far below the median at rush hours: these are holidays and closures (5.8, 5.9).

### 5.4 Calendar effects

In [33]:
idx = y_reg.index
INTRADAY = idx.hour.nunique() > 1
cal = pd.DataFrame({"y": y_reg, "dow": idx.dayofweek, "month": idx.month, "hour": idx.hour})
pivot = cal.pivot_table(index="dow", columns="hour", values="y", aggfunc="mean").reindex(range(7))

fig = go.Figure(go.Heatmap(z=pivot.values, x=[f"{h:02d}h" for h in pivot.columns], y=DAY_LABELS,
                           colorscale=SEQ_BLUE, colorbar=dict(title="Vehicles/hour"), xgap=2, ygap=2,
                           text=(pivot.values / 1000).round(1), texttemplate="%{text}", textfont=dict(size=9),
                           hovertemplate="%{y} %{x}: %{z:,.0f} vehicles/hour<extra></extra>"))
show_fig(fig, "Mean traffic (thousands) by weekday × hour: the 7h and 16h peaks exist only Monday–Friday",
         height=420, yaxis_autorange="reversed")

The same numbers on a **24-hour clock**, one line per weekday: weekdays draw two lobes (morning, afternoon), weekends one midday lobe.

In [34]:
theta = [f"{h:02d}h" for h in range(24)]
fig = go.Figure()
day_colors = [BLUE] * 5 + [ORANGE, RED]
for d in range(7):
    r = pivot.loc[d].values
    fig.add_trace(go.Scatterpolar(r=np.r_[r, r[0]], theta=theta + theta[:1], mode="lines", name=DAY_LABELS[d],
                                  line=dict(color=day_colors[d], width=2.5 if d >= 5 else 1.5),
                                  opacity=1 if d >= 5 else 0.7,
                                  hovertemplate="%{theta}: %{r:,.0f}<extra>" + DAY_LABELS[d] + "</extra>"))
show_fig(fig, "24-hour clock of mean traffic: weekdays (blue) have two lobes, Saturday and Sunday one",
         height=560, legend=dict(orientation="v", x=1.02, y=0.5, yanchor="middle"),
         polar=dict(angularaxis=dict(direction="clockwise", rotation=90, gridcolor=GRID),
                    radialaxis=dict(gridcolor=GRID, tickvals=[2000, 4000, 6000], ticktext=["2k", "4k", "6k"],
                                    angle=45, tickangle=45, tickfont=dict(size=10, color=INK_2))))

In [35]:
mth = pd.DataFrame({"total": day_total, "month": day_total.index.month,
                    "type": np.where(day_total.index.dayofweek >= 5, "Weekend", "Weekday (Mon–Fri)")})
fig = go.Figure()
for name, color in [("Weekday (Mon–Fri)", BLUE), ("Weekend", ORANGE)]:
    part = mth[mth["type"] == name]
    fig.add_trace(box_from_groups(part["total"], part["month"].map(dict(enumerate(MONTH_LABELS, start=1))),
                                  color=color, name=name))
show_fig(fig, "Daily total by month (box = 25–75%, whiskers = 5–95%): the yearly cycle is mild",
         height=430, yaxis_title="Vehicles/day", boxmode="group",
         xaxis=dict(categoryorder="array", categoryarray=MONTH_LABELS))

In [36]:
by = {"hour of day": cal.groupby("hour")["y"].mean(), "weekday": cal.groupby("dow")["y"].mean().rename(dict(enumerate(DAY_LABELS))),
      "month": cal.groupby("month")["y"].mean().rename(dict(enumerate(MONTH_LABELS, start=1)))}
pd.DataFrame({"highest": {k: f"{s.idxmax()} ({s.max():,.0f})" for k, s in by.items()},
              "lowest": {k: f"{s.idxmin()} ({s.min():,.0f})" for k, s in by.items()},
              "highest / lowest": {k: s.max() / s.min() for k, s in by.items()}})

,highest,lowest,highest / lowest
hour of day,"16 (5,742)",3 (382),15.044
weekday,"Fri (3,686)","Sun (2,465)",1.495
month,"Mar (3,458)","Jan (3,091)",1.119


Does the season change the **shape** of a weekday (for example a lower winter morning peak)?

In [37]:
season_of = {12: "Winter (Dec–Feb)", 1: "Winter (Dec–Feb)", 2: "Winter (Dec–Feb)", 3: "Spring (Mar–May)",
             4: "Spring (Mar–May)", 5: "Spring (Mar–May)", 6: "Summer (Jun–Aug)", 7: "Summer (Jun–Aug)",
             8: "Summer (Jun–Aug)", 9: "Fall (Sep–Nov)", 10: "Fall (Sep–Nov)", 11: "Fall (Sep–Nov)"}
wk = y_reg[(idx.dayofweek < 5) & hol_name.eq("None").to_numpy()]
season_prof = wk.groupby([wk.index.month.map(season_of), wk.index.hour]).mean().unstack(0)

fig = go.Figure()
for name, color in zip(["Winter (Dec–Feb)", "Spring (Mar–May)", "Summer (Jun–Aug)", "Fall (Sep–Nov)"],
                       [BLUE, AQUA, ORANGE, YELLOW]):
    fig.add_trace(go.Scatter(x=season_prof.index, y=season_prof[name], mode="lines", name=name,
                             line=dict(color=color, width=2.5), hovertemplate="%{y:,.0f}<extra>" + name + "</extra>"))
peak = season_prof.loc[7]
show_fig(fig, f"Weekday profile by season: same shape all year, the winter 7h peak is "
              f"{1 - peak['Winter (Dec–Feb)'] / peak.drop('Winter (Dec–Feb)').mean():.0%} lower",
         height=420, xaxis=dict(title="Hour of day", dtick=2), yaxis_title="Vehicles/hour (mean)", hovermode="x unified")

**Observation.** The busiest hour (16h, ≈ 5,740 vehicles) carries **about 15 times** the quietest (3h, ≈ 380), Friday is 1.5 times busier than Sunday, while the busiest month (March) exceeds the quietest (January) by only **about 12%**. The 6–7h and 15–17h peaks exist only Monday to Friday (commuting); weekends have one broad peak around noon. The weekday shape is almost the same in every season; the clearest difference is a winter morning peak about 6% lower. → The **daily and weekly cycles** dominate and must be encoded (hour, weekday, `lag_24`, `lag_168`); the yearly cycle is a small correction (month as sin/cos).

### 5.5 Seasonal decomposition

**Idea.** A series can be read as the **sum of layers**:

$$y_t = \underbrace{T_t}_{\text{trend}} + \underbrace{S^{(24)}_t}_{\text{daily season}} + \underbrace{S^{(168)}_t}_{\text{weekly season}} + \underbrace{R_t}_{\text{residual}}$$

- **Trend $T_t$**: the slowly moving base level (weeks to months).
- **Daily season $S^{(24)}_t$**: the wave that repeats every 24 hours (rush hours, quiet nights).
- **Weekly season $S^{(168)}_t$**: the extra offset that repeats every 168 hours = 7 days (weekdays high, weekends low).
- **Residual $R_t$**: whatever is left: events, weather, noise.

`MSTL` (*Multiple Seasonal-Trend decomposition using LOESS*) extracts the layers in turn: it smooths out the 24-hour season, removes it, extracts the 168-hour season, and iterates a few times; the smooth remainder is the trend. Plain `STL` handles one cycle only; MSTL handles **nested cycles** like these.

Two numbers say which layer matters:
- **share of variance** $= \text{Var}(\text{component}) / \text{Var}(y)$;
- **seasonal strength** (Wang, Smith & Hyndman, 2006) $F_S = \max\left(0,\ 1 - \text{Var}(R) / \text{Var}(S + R)\right)$: close to 1 means the season dominates the noise.

In [38]:
periods = [p for p in SEASONAL_PERIODS if 2 * p < len(y_filled)]
decomposition = MSTL(y_filled, periods=periods, windows=[p + 1 for p in periods]).fit()
seasonal = decomposition.seasonal                             # one column per period: seasonal_24, seasonal_168
components = pd.concat([y_filled.rename("observed"), decomposition.trend.rename("trend"), seasonal,
                        decomposition.resid.rename("residual")], axis=1)
share = (components.drop(columns="observed").var() / components["observed"].var() * 100).rename("% of variance")
strength = pd.Series({c: max(0, 1 - components["residual"].var() / (components["residual"] + components[c]).var())
                      for c in seasonal.columns}, name="seasonal strength (0–1)")
pd.concat([share, strength], axis=1)

,% of variance,seasonal strength (0–1)
trend,1.155,NaN
seasonal_24,79.042,0.953
seasonal_168,15.365,0.798
residual,3.888,NaN


In [39]:
recent = components.loc["2018-03-05":"2018-03-25"]
fig = make_subplots(rows=len(recent.columns), cols=1, shared_xaxes=True, vertical_spacing=0.035,
                    subplot_titles=[c + (f": {share[c]:.0f}% of variance" if c in share else "") for c in recent.columns])
for i, c in enumerate(recent.columns, start=1):
    color = QUALITATIVE[(i - 1) % 6]
    rgba = "rgba({},{},{},0.12)".format(*(int(color[k:k + 2], 16) for k in (1, 3, 5)))
    fig.add_trace(go.Scatter(x=recent.index, y=recent[c], mode="lines", line=dict(color=color, width=1.6),
                             fill="tozeroy" if i > 2 else None, fillcolor=rgba, showlegend=False,
                             hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f}<extra>" + c + "</extra>"), row=i, col=1)
show_fig(fig, "MSTL over 3 weeks of March 2018: observed = trend + daily + weekly + residual", height=860)

**How to read it.** Adding the four lower rows at any instant gives the top row. The trend barely moves; the daily row repeats the same wave every day; the weekly row rises Monday–Friday and drops on Saturday and Sunday mornings; the residual is small, with a few spikes on unusual hours.

Because the seasons repeat, each can be averaged into **one representative cycle**:

In [40]:
s24 = components["seasonal_24"].groupby(components.index.hour).mean()
s168 = components["seasonal_168"].groupby(components.index.dayofweek * 24 + components.index.hour).mean()

fig = make_subplots(rows=1, cols=2, column_widths=[0.32, 0.68], horizontal_spacing=0.08,
                    subplot_titles=("One daily cycle (24h component)", "One weekly cycle (168h component)"))
fig.add_trace(go.Bar(x=s24.index, y=s24.values, marker_color=np.where(s24.values >= 0, AQUA, "#a9dcc8"),
                     hovertemplate="%{x}h: %{y:+,.0f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=s168.index, y=s168.values, mode="lines", line=dict(color=YELLOW, width=2),
                         fill="tozeroy", fillcolor="rgba(237,161,0,0.15)",
                         hovertemplate="hour %{x} of the week: %{y:+,.0f}<extra></extra>"), row=1, col=2)
fig.update_xaxes(title_text="Hour of day", dtick=6, row=1, col=1)
fig.update_xaxes(tickvals=np.arange(12, 168, 24), ticktext=DAY_LABELS, row=1, col=2)
for d in range(1, 7):
    fig.add_vline(x=d * 24, line_color=GRID, line_width=1, row=1, col=2)
fig.update_yaxes(title_text="Deviation from trend (vehicles/hour)", col=1)
show_fig(fig, f"The two seasonal shapes: the daily cycle swings {s24.max() - s24.min():,.0f} vehicles, "
              "the weekly cycle cuts Saturday and Sunday mornings", height=420, showlegend=False)

**Observation.** The daily season explains about **79%** of the variance, the weekly season about **15%**, the trend about **1%** and the residual about **4%**; the seasonal strengths are 0.95 (daily) and 0.80 (weekly). In plain words: **knowing the hour and the weekday already predicts about 95% of the variation**; only the residual (events, weather) is hard. The trend is almost flat, so no differencing is needed to remove a trend; what a model must capture is the two nested cycles.

### 5.6 Autocorrelation (ACF / PACF) and stationarity

**Autocorrelation** at lag $k$ measures how much the current value resembles the value $k$ steps earlier: $\text{ACF}(k) = \text{corr}(y_t, y_{t-k})$. Before the formal plot, here is that correlation drawn directly: $y_t$ against $y_{t-k}$ for a few $k$. The closer the cloud is to the diagonal, the better the value $k$ hours ago predicts the present.

In [41]:
lag_list = [1, 3, 12, 24, 168]
lag_names = {1: "1h ago", 3: "3h ago", 12: "12h ago", 24: "yesterday", 168: "last week"}
edges = np.linspace(0, 7400, 61)
centers = (edges[:-1] + edges[1:]) / 2
fig = make_subplots(rows=1, cols=len(lag_list), shared_yaxes=True, horizontal_spacing=0.025,
                    subplot_titles=[f"lag {k}: {lag_names[k]}<br>r = {y_filled.autocorr(k):.2f}" for k in lag_list])
for c, k in enumerate(lag_list, start=1):
    pair = pd.DataFrame({"now": y_filled, "past": y_filled.shift(k)}).dropna()
    H, _, _ = np.histogram2d(pair["past"], pair["now"], bins=[edges, edges])
    fig.add_trace(go.Heatmap(z=np.log1p(H.T), x=centers, y=centers, colorscale=SEQ_BLUE, showscale=False,
                             customdata=H.T, hovertemplate="past %{x:,.0f}, now %{y:,.0f}: %{customdata:,.0f} hours<extra></extra>"),
                  row=1, col=c)
    fig.add_trace(go.Scatter(x=[0, 7400], y=[0, 7400], mode="lines", line=dict(color=RED, dash="dot", width=1),
                             hoverinfo="skip"), row=1, col=c)
    fig.update_xaxes(title_text="y(t−k)", range=[0, 7400], showticklabels=False, row=1, col=c)
fig.update_yaxes(title_text="y(t)", range=[0, 7400], col=1)
show_fig(fig, "Lag plots (density): 12 hours ago is a poor guide, the same hour last week is the best",
         height=380, showlegend=False, margin=dict(t=110))

- **ACF** is the $r$ above computed for every lag $k = 0, 1, 2, \dots$; peaks that repeat reveal the cycles.
- **PACF** (*partial* autocorrelation) asks a finer question: *once lags $1, \dots, k-1$ are known, does lag $k$ still add information?* ACF at lag 2 is high mostly because lag 2 "passes through" lag 1; PACF removes that indirect part. It gives the order $p$ of an autoregressive model: the PACF cuts off after lag $p$.
- Bars outside the dotted band ($\pm 1.96/\sqrt{n}$) are significant at 5%.

In [42]:
N_ACF = min(3 * max(SEASONAL_PERIODS), len(y_filled) // 4)
N_PACF = min(3 * min(SEASONAL_PERIODS), len(y_filled) // 4)
acf_vals = acf(y_filled, nlags=N_ACF, fft=True)
pacf_vals = pacf(y_filled, nlags=N_PACF, method="ywm")
ci = 1.96 / np.sqrt(len(y_filled))

lags_axis = np.arange(N_ACF + 1)
acf_colors = np.where(lags_axis % 168 == 0, RED, np.where(lags_axis % 24 == 0, ORANGE, BLUE))
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.14,
                    subplot_titles=(f"ACF up to lag {N_ACF}: orange = multiples of 24h, red = multiples of 168h",
                                    f"PACF up to lag {N_PACF}"))
fig.add_trace(go.Bar(x=lags_axis, y=acf_vals, marker_color=acf_colors, marker_line_width=0,
                     hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=np.arange(N_PACF + 1), y=pacf_vals, marker_color=np.where(np.abs(pacf_vals) > ci, ORANGE, MUTED),
                     hovertemplate="lag %{x}: %{y:.3f}<extra></extra>"), row=2, col=1)
for r in (1, 2):
    for s in (ci, -ci):
        fig.add_hline(y=s, line_dash="dot", line_color=MUTED, row=r, col=1)
for lag in (12, 24, 168, 336):
    fig.add_annotation(x=lag, y=acf_vals[lag], text=f"lag {lag}: {acf_vals[lag]:.2f}", showarrow=True,
                       arrowcolor=MUTED, ay=30 if acf_vals[lag] < 0 else -30, font_color=INK_2, row=1, col=1)
fig.update_xaxes(title_text="Lag (hours)", row=2, col=1)
fig.update_xaxes(tickvals=np.arange(0, N_ACF + 1, 24), row=1, col=1)
show_fig(fig, "Autocorrelation peaks every 24h and is highest at 168h: the daily and weekly cycles",
         height=680, showlegend=False, bargap=0.25)
pd.DataFrame({"ACF": [acf_vals[k] for k in [*LAGS, 12]],
              "PACF": [pacf_vals[k] if k <= N_PACF else np.nan for k in [*LAGS, 12]]}, index=pd.Index([*LAGS, 12], name="lag"))

,ACF,PACF
lag,,
1,0.914,0.914
2,0.725,-0.668
3,0.511,0.185
24,0.866,-0.100
168,0.944,NaN
12,-0.616,0.154


**Observation.** The ACF is a **wave with a 24-hour period**: negative at lag 12 (−0.62, day against night), high at lag 24 (0.87) and highest at **lag 168 (0.94)**. The weekly peaks (red) rise above the daily peaks next to them: "same hour last week" resembles the present more than "same hour yesterday", because yesterday may have been a weekend day. The ACF decays very slowly over 15 days, the signature of strong seasonality. The PACF is large at lag 1 (0.91) and strongly negative at lag 2 (−0.67); after that a cluster of positive values around lags 18–23 and a negative one at lag 25 remain, traces of the daily cycle that a short autoregression does not capture.

→ `LAGS = [1, 2, 3, 24, 168]` (short-term memory + same hour yesterday + same hour last week). For an ARIMA-type model, the PACF points to an autoregressive part of order about 2–3, with the two cycles handled separately.

**Stationarity.** ARIMA models assume a **stationary** series (constant mean and variance) after differencing $d$ times. Two complementary tests:

| Test | Null hypothesis $H_0$ | "Stationary" when |
|---|---|---|
| **ADF** (augmented Dickey–Fuller) | the series has a unit root (**not stationary**) | p < 0.05 (reject $H_0$) |
| **KPSS** | the series is **stationary** around its mean | p > 0.05 (keep $H_0$) |

In [43]:
def stationarity(series, name):
    adf_stat, adf_p, *_ = adfuller(series, maxlag=48, autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    verdict = "stationary (both tests)" if adf_p < 0.05 and kpss_p > 0.05 else (
        "no unit root, level drifts (ADF yes, KPSS no)" if adf_p < 0.05 else "not stationary")
    return {"series": name, "ADF stat": adf_stat, "ADF p": adf_p, "KPSS stat": kpss_stat, "KPSS p": kpss_p, "verdict": verdict}


deseasonalized = components["trend"] + components["residual"]
pd.DataFrame([stationarity(y_filled, TARGET),
              stationarity(deseasonalized, "deseasonalized (trend + residual)"),
              stationarity(y_filled.diff(168).dropna(), "weekly difference y(t) − y(t−168)")]).set_index("series")

,ADF stat,ADF p,KPSS stat,KPSS p,verdict
series,,,,,
traffic_volume,-21.809,0.000,0.559,0.028,"no unit root, level drifts (ADF yes, KPSS no)"
deseasonalized (trend + residual),-14.221,0.000,0.750,0.010,"no unit root, level drifts (ADF yes, KPSS no)"
weekly difference y(t) − y(t−168),-16.065,0.000,0.014,0.100,stationary (both tests)


In [44]:
ROLL = 24 * 28
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=("Original series", "Deseasonalized (trend + residual)"))
for c, s in enumerate([y_filled, deseasonalized], start=1):
    m = s.rolling(ROLL, center=True).mean().iloc[::24].dropna()   # one point per day keeps the figure light
    sd = s.rolling(ROLL, center=True).std().iloc[::24].dropna()
    band(fig, m.index.to_numpy(), (m - sd).values, (m + sd).values, "rgba(42,120,214,0.15)", "±1 std", row=1, col=c)
    fig.add_trace(go.Scatter(x=m.index, y=m.values, mode="lines", line=dict(color=BLUE, width=2.5), showlegend=False,
                             hovertemplate="%{x|%d/%m/%Y}: %{y:,.0f}<extra>4-week mean</extra>"), row=1, col=c)
fig.update_yaxes(title_text="Vehicles/hour", col=1)
show_fig(fig, "4-week rolling mean ± std: level and spread stay roughly constant, no trend to difference away",
         height=400)

**Observation.** The 4-week rolling mean and standard deviation stay almost flat across 2016–2018, both for the raw and for the deseasonalized series (whose band is much narrower).
- **ADF** rejects a unit root for all three series (p ≈ 0): there is no random-walk trend.
- **KPSS** still rejects stationarity at 5% for the raw (p = 0.03) and deseasonalized (p = 0.01) series: the level drifts slowly with the seasons (low in December–January), and KPSS is very sensitive with $n$ ≈ 24,000 hours.
- The **weekly difference** $y_t - y_{t-168}$ passes both tests.

→ For ARIMA: no ordinary differencing ($d = 0$). The seasons must be handled explicitly, either by a weekly difference or by **Fourier terms** (sin/cos of the 168-hour period) as regressors. A seasonal SARIMA with period $m = 168$ is too heavy to estimate on ~17,000 training hours; section 7.1 uses the Fourier approach.

### 5.7 Covariates: does the weather move the target?

Raw correlations are confounded by the calendar (fog and mist happen at dawn, when traffic is low anyway). The table compares each numeric covariate with the raw target and with the **deviation from the usual value for that hour of the week** (`resid` from 4.5).

In [45]:
win = eda
pd.DataFrame({
    f"r with {TARGET}": [win[c].corr(y_reg) for c in NUM_EXOG],
    "r with deviation from seasonal profile": [win[c].corr(resid) for c in NUM_EXOG],
    f"r with deviation, covariate {HORIZON} step(s) earlier": [win[c].shift(HORIZON).corr(resid) for c in NUM_EXOG],
}, index=NUM_EXOG)

,r with traffic_volume,r with deviation from seasonal profile,"r with deviation, covariate 1 step(s) earlier"
temp_c,0.127,0.098,0.102
rain_1h,0.007,-0.004,-0.004
snow_1h,-0.000,-0.006,-0.002
clouds_all,0.118,-0.084,-0.078
n_weather,-0.044,-0.128,-0.129


For the weather categories, each hour is compared with the **median of the same weekday-hour** on non-holiday days, as a percentage:

$$\text{deviation}_t = \frac{y_t - \text{median}(y \mid \text{weekday}, \text{hour})}{\text{median}(y \mid \text{weekday}, \text{hour})}$$

Only daytime hours 6h–20h (the denominator is large enough) and observed hours (not interpolated) are used.

In [46]:
base = eda[hol_name.eq("None").to_numpy() & y_reg.notna().to_numpy()].copy()
base["how"] = base.index.dayofweek * 24 + base.index.hour
base["expected"] = base.groupby("how")[TARGET].transform("median")
day = base[(base.index.hour >= 6) & (base.index.hour <= 20)]
day = day.assign(rel=(day[TARGET] / day["expected"] - 1) * 100)


def effect_table(groups, min_n=30):
    """Mean % deviation, its 95% confidence half-width and the count, per group with at least min_n hours."""
    t = day.groupby(groups)["rel"].agg(["mean", "std", "count"]).rename(columns={"count": "n"})
    t = t[t["n"] >= min_n]
    t["ci95"] = 1.96 * t["std"] / np.sqrt(t["n"])
    return t.sort_values("mean")


eff = effect_table(day["weather_main"])
print(f"{len(day):,} daytime hours compared")
eff

13,948 daytime hours compared


,mean,std,n,ci95
weather_main,,,,
Snow,-10.633,17.301,879,1.144
Thunderstorm,-3.041,10.333,210,1.398
Mist,-1.872,11.213,907,0.730
Rain,-1.808,10.999,1645,0.532
Haze,-0.669,9.688,290,1.115
Clear,-0.425,9.696,4172,0.294
Drizzle,-0.263,7.813,196,1.094
Clouds,-0.170,9.920,5577,0.260
Fog,-0.021,7.382,66,1.781


In [47]:
fig = go.Figure(go.Bar(x=eff["mean"], y=eff.index, orientation="h",
                       marker_color=np.where(eff["mean"] < -5, RED, np.where(eff["mean"] < 0, "#f2a3a2", BLUE)),
                       error_x=dict(type="data", array=eff["ci95"], color=INK_2, thickness=1.2, width=4),
                       hovertemplate="%{y}: %{x:+.1f}% vs. same weekday-hour<extra></extra>"))
fig.add_trace(go.Scatter(x=np.minimum(eff["mean"], 0) - eff["ci95"] - 0.3, y=eff.index, mode="text",
                         text=[f"{m:+.1f}%  (n={n:,})" for m, n in zip(eff["mean"], eff["n"])],
                         textposition="middle left", textfont_color=INK_2, hoverinfo="skip"))
fig.add_vline(x=0, line_color=INK_2, line_width=1)
show_fig(fig, f"Only snow cuts daytime traffic clearly ({eff.loc['Snow', 'mean']:+.1f}%); other weather types stay within ±3%",
         height=430, showlegend=False, xaxis_title="Mean deviation from the same weekday-hour (%, ±95% CI)",
         xaxis_range=[eff["mean"].min() * 1.6 - 1, 3])

`weather_main` is coarse: "Snow" mixes light flurries and blizzards. Splitting rain and snow by `weather_description`:

In [48]:
rs = day[day["weather_main"].isin(["Rain", "Snow", "Drizzle", "Thunderstorm"])]
eff_desc = effect_table(rs["weather_description"], min_n=20)
group_of = rs.groupby("weather_description")["weather_main"].first()

fig = go.Figure()
for g_name, color in {"Rain": BLUE, "Snow": RED, "Drizzle": AQUA, "Thunderstorm": YELLOW}.items():
    part = eff_desc[group_of.reindex(eff_desc.index) == g_name]
    fig.add_trace(go.Scatter(x=part["mean"], y=part.index, mode="markers", name=g_name,
                             marker=dict(color=color, size=np.clip(np.sqrt(part["n"]) * 1.4, 7, 30),
                                         line=dict(color="#ffffff", width=1.5)),
                             error_x=dict(type="data", array=part["ci95"], color=color, thickness=1.2, width=0),
                             customdata=part["n"], hovertemplate="%{y}: %{x:+.1f}% (n=%{customdata:,})<extra></extra>"))
fig.add_vline(x=0, line_color=INK_2, line_width=1)
show_fig(fig, f"Intensity matters: heavy snow cuts traffic by {-eff_desc.loc['heavy snow', 'mean']:.0f}%, "
              f"light snow by {-eff_desc.loc['light snow', 'mean']:.0f}% (dot size ∝ √n)",
         height=520, xaxis_title="Mean deviation from the same weekday-hour (%)",
         yaxis=dict(categoryorder="array", categoryarray=list(eff_desc.index)))

In [49]:
dd = pd.DataFrame({"total": day_total, "temp": eda["temp_c"].resample("D").mean()})[~is_hol_day]
dd["type"] = np.where(dd.index.dayofweek >= 5, "Weekend", "Weekday (Mon–Fri)")
fig = go.Figure()
for name, color in [("Weekday (Mon–Fri)", BLUE), ("Weekend", ORANGE)]:
    part = dd[dd["type"] == name]
    fig.add_trace(go.Scatter(x=part["temp"], y=part["total"], mode="markers", name=name,
                             marker=dict(color=color, size=7, opacity=0.6, line=dict(color="#ffffff", width=0.8)),
                             hovertemplate="%{x:.1f} °C: %{y:,.0f} vehicles/day<extra></extra>"))
    trend = part.groupby(pd.cut(part["temp"], np.arange(-30, 40, 5)), observed=True)["total"].median()   # median per 5 °C bin
    fig.add_trace(go.Scatter(x=[b.mid for b in trend.index], y=trend.values, mode="lines", showlegend=False,
                             line=dict(color=color, width=3), hoverinfo="skip"))
r_wd = dd.loc[dd["type"] == "Weekday (Mon–Fri)", ["temp", "total"]].corr().iloc[0, 1]
show_fig(fig, f"Daily temperature vs. daily total (holidays excluded): weak link, r = {r_wd:.2f} on weekdays",
         height=440, xaxis_title="Daily mean temperature (°C)", yaxis_title="Vehicles/day")

In [50]:
snow_rel = day.loc[day["weather_main"] == "Snow", "rel"]
heavy = day["weather_description"].str.contains("heavy", case=False)
print(f"snow hours: {snow_rel.mean():+.1f}% (n={len(snow_rel):,}) | 'heavy' rain/snow hours: "
      f"{day.loc[heavy, 'rel'].mean():+.1f}% (n={heavy.sum():,}) | clear hours: "
      f"{day.loc[day['weather_main'] == 'Clear', 'rel'].mean():+.1f}%")

snow hours: -10.6% (n=879) | 'heavy' rain/snow hours: -13.6% (n=201) | clear hours: -0.4%


**Observation.** After removing the hour-of-week effect, most weather types move traffic by less than 2–3%: correlations of the numeric covariates with the deviation stay below 0.1 in absolute value. The clear exception is **snow: −10.6%** over 879 daytime hours, and −13.6% for any "heavy" rain or snow. The effect **grows with intensity**: light snow ≈ −7%, snow ≈ −27%, **heavy snow ≈ −43%**; heavy rain ≈ −8% while light and moderate rain barely matter. Temperature is only weakly related to the daily total (r = 0.14 on weekdays); only days below about −15 °C drop, and they usually coincide with snow.

→ The weather matters **rarely but strongly, and non-linearly**. `weather_description` (which carries the intensity) is kept as a categorical covariate; tree models are expected to use it better than a linear model. Covariates enter the features with a 1-hour delay (6.2), so the model uses the weather observed at forecast time, not a weather forecast.

### 5.8 Question 1: How much lower is traffic on holidays, and is every holiday alike?

In [51]:
day_type = np.select([hol_name.ne("None").to_numpy(), y_reg.index.dayofweek >= 5], ["Holiday", "Weekend"], "Weekday (Mon–Fri)")
order = ["Weekday (Mon–Fri)", "Weekend", "Holiday"]
prof = y_reg.groupby([day_type, y_reg.index.hour]).mean().unstack(0)
daily_by_type = (pd.DataFrame({"total": day_total, "type": np.select([is_hol_day.to_numpy(), day_total.index.dayofweek >= 5],
                                                                       ["Holiday", "Weekend"], "Weekday (Mon–Fri)")})
                   .groupby("type")["total"].mean().reindex(order))
holiday_drop = 1 - daily_by_type["Holiday"] / daily_by_type["Weekday (Mon–Fri)"]
weekend_drop = 1 - daily_by_type["Weekend"] / daily_by_type["Weekday (Mon–Fri)"]

fig = go.Figure()
band(fig, prof.index.to_numpy(), prof["Holiday"].values, prof["Weekday (Mon–Fri)"].values, "rgba(27,175,122,0.12)", "lost")
for name, color in zip(order, [BLUE, ORANGE, AQUA]):
    fig.add_trace(go.Scatter(x=prof.index, y=prof[name], mode="lines+markers", name=name,
                             line=dict(color=color, width=2.5), marker=dict(size=7, line=dict(color="#ffffff", width=1.5)),
                             hovertemplate="%{y:,.0f}<extra>" + name + "</extra>"))
show_fig(fig, f"Hourly profile: holidays carry {holiday_drop:.0%} less daily traffic than weekdays, weekends {weekend_drop:.0%} less",
         height=430, xaxis=dict(title="Hour of day", dtick=2), yaxis_title="Vehicles/hour (mean)", hovermode="x unified")
daily_by_type.rename("mean vehicles/day").to_frame().assign(
    **{"vs. weekday": lambda d: (d.iloc[:, 0] / d.iloc[0, 0] - 1).map("{:+.1%}".format)})

,mean vehicles/day,vs. weekday
type,,
Weekday (Mon–Fri),"86,656.972",+0.0%
Weekend,"64,092.214",-26.0%
Holiday,"62,317.482",-28.1%


An average hides differences between holidays. For **each holiday**, the actual daily total is compared with an ordinary day: the median total of the same weekday in the 4 weeks before and after (holidays excluded).

In [52]:
rows = []
for d in day_total.index[is_hol_day]:
    same_dow = [d + pd.Timedelta(weeks=k) for k in range(-4, 5) if k != 0]
    same_dow = [x for x in same_dow if x in day_total.index and not is_hol_day[x]]
    rows.append({"date": d, "holiday": holiday_by_day[d], "actual": day_total[d], "expected": day_total[same_dow].median()})
hol = pd.DataFrame(rows).assign(change=lambda d: d["actual"] / d["expected"] - 1).sort_values("change")
hol["label"] = hol["holiday"] + " · " + hol["date"].dt.strftime("%a %d/%m/%y")

fig = go.Figure()
for _, r in hol.iterrows():
    fig.add_shape(type="line", x0=r["actual"], x1=r["expected"], y0=r["label"], y1=r["label"], line=dict(color=GRID, width=4))
fig.add_trace(go.Scatter(x=hol["expected"], y=hol["label"], mode="markers", name="ordinary day (same weekday, ±4 weeks)",
                         marker=dict(color=MUTED, size=10), hovertemplate="expected %{x:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=hol["actual"], y=hol["label"], mode="markers+text", name="the holiday",
                         marker=dict(color=AQUA, size=12, line=dict(color="#ffffff", width=1.5)),
                         text=[f"{v:+.0%}" for v in hol["change"]], textfont_color=INK_2,
                         textposition=np.where(hol["change"] < 0, "middle left", "middle right"),
                         hovertemplate="actual %{x:,.0f}<extra></extra>"))
show_fig(fig, f"Each holiday vs. an ordinary day: from {hol['change'].min():+.0%} ({hol.iloc[0]['holiday']}) "
              f"to {hol['change'].max():+.0%} ({hol.iloc[-1]['holiday']})",
         height=760, margin=dict(l=280), xaxis=dict(title="Vehicles/day", range=[0, hol["expected"].max() * 1.08]),
         yaxis=dict(autorange="reversed", tickfont=dict(size=11)))

In [53]:
by_name = hol.assign(major=hol["holiday"].isin(MAJOR_HOLIDAYS)).groupby("holiday").agg(
    mean_change=("change", "mean"), min_change=("change", "min"), max_change=("change", "max"),
    days=("change", "size"), in_MAJOR_HOLIDAYS=("major", "first")).sort_values("mean_change")
by_name.style.format({c: "{:+.1%}" for c in ["mean_change", "min_change", "max_change"]})

,mean_change,min_change,max_change,days,in_MAJOR_HOLIDAYS
holiday,,,,,
Thanksgiving Day,-47.7%,-49.0%,-46.4%,2,True
New Years Day,-44.1%,-49.0%,-35.5%,3,True
Independence Day,-43.7%,-48.4%,-40.8%,3,True
Christmas Day,-39.0%,-45.0%,-33.1%,2,True
Memorial Day,-38.3%,-40.0%,-37.4%,3,True
Labor Day,-33.7%,-34.6%,-33.1%,3,True
Martin Luther King Jr Day,-17.7%,-21.5%,-13.8%,2,False
Washingtons Birthday,-15.8%,-28.3%,-8.8%,3,False
Veterans Day,-3.1%,-3.8%,-2.4%,2,False


**Conclusion.** On average a **holiday carries about 28% less traffic than a weekday**, close to a weekend (−26%), and the holiday morning peak is only about half of a weekday's. But the holidays are far from alike:
- the big ones (New Year, Thanksgiving, Independence Day, Christmas, Memorial Day, Labor Day) lose **33–49%**;
- Martin Luther King Jr Day and Washington's Birthday lose 9–28% (many employers stay open);
- Veterans Day and Columbus Day lose almost nothing (−4% to 0%);
- **State Fair** days (the Minnesota State Fair, not a day off) even gain 1–5%.

→ A single `is_holiday` flag mixes opposite effects. The features therefore get both `is_holiday` and `is_major_holiday` (the six holidays listed in `MAJOR_HOLIDAYS`), both known in advance from the calendar.

### 5.9 Question 2: Which outlier detector finds a road closure: rolling mean/std or rolling median/MAD?

Section 4.5 used a robust z-score. Would the textbook version, $z_t = (r_t - \text{mean}_{168}(r)) / \text{std}_{168}(r)$ on the same residual and window, find the same events? A **road-closure hour** is defined as a daytime hour (7h–19h) with fewer than 300 vehicles while the usual value for that hour of the week is above 3,000.

In [54]:
z_std = (resid - window.mean()) / window.std()
flag_std = z_std.abs() > Z_THRESHOLD
closure = (y_reg < 300) & (profile > 3000) & y_reg.index.hour.isin(range(7, 20))

pd.DataFrame({
    "flagged hours": [int(flag_std.sum()), int(is_outlier.sum())],
    "% of hours": [flag_std.mean() * 100, is_outlier.mean() * 100],
    f"closure hours found (of {int(closure.sum())})": [int((flag_std & closure).sum()), int((is_outlier & closure).sum())],
    "flagged hours on holidays": [int((flag_std & hol_name.ne("None")).sum()), int((is_outlier & hol_name.ne("None")).sum())],
}, index=["rolling mean / std", "rolling median / MAD (4.5)"])

,flagged hours,% of hours,closure hours found (of 14),flagged hours on holidays
rolling mean / std,248,1.029,0,52
rolling median / MAD (4.5),1016,4.216,14,184


In [55]:
seg = slice("2016-07-04", "2016-07-31")
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.65, 0.35],
                    subplot_titles=("Actual vs. usual traffic", "z-score of the residual: red = median/MAD, yellow dotted = mean/std"))
fig.add_trace(go.Scatter(x=profile[seg].index, y=profile[seg].values, mode="lines", name="usual (median of the same hour of the week)",
                         line=dict(color=MUTED, width=1.5, dash="dot"), hovertemplate="%{y:,.0f}<extra>usual</extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=y_reg[seg].index, y=y_reg[seg].values, mode="lines", name="actual",
                         line=dict(color=BLUE, width=1.8), hovertemplate="%{y:,.0f}<extra>actual</extra>"), row=1, col=1)
for name, flag, color, symbol in [("median/MAD", is_outlier, RED, "circle"), ("mean/std", flag_std, ORANGE, "x")]:
    pts = y_reg[seg][flag[seg]]
    fig.add_trace(go.Scatter(x=pts.index, y=pts.values, mode="markers", name=f"{name}: {len(pts)} hours flagged",
                             marker=dict(color=color, size=10, symbol=symbol, line=dict(color="#ffffff", width=1.5)),
                             hovertemplate="%{x|%a %d/%m %H:%M}: %{y:,.0f}<extra>" + name + "</extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=z_robust[seg].index, y=z_robust[seg].clip(-15, 15).values, mode="lines", showlegend=False,
                         line=dict(color=RED, width=1.6), hovertemplate="%{y:.1f}<extra>z median/MAD</extra>"), row=2, col=1)
fig.add_trace(go.Scatter(x=z_std[seg].index, y=z_std[seg].clip(-15, 15).values, mode="lines", showlegend=False,
                         line=dict(color=YELLOW, width=1.6, dash="dot"), hovertemplate="%{y:.1f}<extra>z mean/std</extra>"), row=2, col=1)
for s in (Z_THRESHOLD, -Z_THRESHOLD):
    fig.add_hline(y=s, line_dash="dot", line_color=INK_2, row=2, col=1)
fig.update_yaxes(title_text="Vehicles/hour", row=1, col=1)
fig.update_yaxes(title_text="z (clipped ±15)", row=2, col=1)
show_fig(fig, f"July 2016 road closure: median/MAD flags {int((is_outlier & closure).sum())} of {int(closure.sum())} "
              f"closure hours, mean/std flags {int((flag_std & closure).sum())}",
         height=660, hovermode="x unified", margin=dict(t=130), legend=dict(y=1.08))

In [56]:
flags_per_day = is_outlier.resample("D").sum()
snow_day = eda["weather_main"].eq("Snow").resample("D").sum() >= 6
kind = np.select([is_hol_day.reindex(flags_per_day.index).to_numpy(), snow_day.reindex(flags_per_day.index).to_numpy()],
                 ["Holiday", "Snow day (≥ 6 snowy hours)"], "Other day")
fig = go.Figure()
for name, color in [("Holiday", AQUA), ("Snow day (≥ 6 snowy hours)", BLUE), ("Other day", MUTED)]:
    m = kind == name
    fig.add_trace(go.Bar(x=flags_per_day.index[m], y=flags_per_day.values[m], name=name, marker_color=color, marker_line_width=0,
                         hovertemplate="%{x|%a %d/%m/%Y}: %{y} flagged hours<extra>" + name + "</extra>"))
show_fig(fig, "Flagged hours per day: the tallest bars are holidays and snow days, i.e. real events",
         height=420, yaxis_title="Flagged hours in the day", barmode="overlay", bargap=0)

In [57]:
top = flags_per_day.sort_values(ascending=False).head(10)
pd.DataFrame({"holiday": [holiday_by_day.get(d, "None") for d in top.index],
              "snowy hours": eda["weather_main"].eq("Snow").resample("D").sum().reindex(top.index).values,
              "flagged hours": top.values}, index=top.index.strftime("%a %d/%m/%Y"))

,holiday,snowy hours,flagged hours
timestamp,,,
Thu 11/01/2018,None,12,17
Mon 19/02/2018,Washingtons Birthday,11,15
Mon 29/05/2017,Memorial Day,0,15
Mon 22/01/2018,None,19,15
Wed 04/07/2018,Independence Day,0,14
Sat 23/07/2016,None,0,14
Thu 23/11/2017,Thanksgiving Day,0,14
Sat 14/04/2018,None,20,14
Tue 23/01/2018,None,3,13


**Conclusion.** When the road is closed for many consecutive hours, those hours **inflate the standard deviation of the window**, so their own z-score shrinks and they hide themselves (*masking*). On 23–24 July 2016 the robust z-score falls to about −8, while the mean/std z-score barely touches −3: the median/MAD detector finds every daytime closure hour, the mean/std detector none. Median and MAD ignore a few dozen extreme points, which is exactly what is needed here. The days with the most flagged hours are **holidays and snowstorms**: real events that the model has to live with.

**Decision.** The target values are **not removed or replaced**: they are real (closure, holiday, storm), unlike the 0 K temperature. Cleaning them in the test period would make the scores look artificially good. `is_outlier` is kept only to analyse errors (8.5); it is **not** a feature, because it is computed with a centred window that looks into the future.

<a id="s6"></a>
## 6. Data preparation

### 6.1 Cleaning: regular calendar, errors to NaN, short gaps interpolated

1. merge duplicated timestamps and reindex at `FREQ` (`to_regular`, section 4.2);
2. keep `WINDOW_START`–`WINDOW_END`;
3. impossible values → `NaN` (`VALID_RANGES`, section 4.4);
4. flag `is_imputed = 1` where the target was missing, then fill **only gaps of at most `MAX_INTERP_GAP` steps**: numeric columns by time-linear interpolation (traffic and weather change continuously, and forward fill would "freeze" a rush hour that is rising or falling), categorical columns by forward fill (the last known weather is the best guess, and two labels cannot be averaged). `interpolate(limit=n)` alone would also fill the first n steps of *longer* gaps; the mask below prevents that.
5. domain rule (4.6): spread each holiday name over its whole day.

In [58]:
def short_gap_mask(s, max_len):
    """True on NaN cells that belong to a run of at most max_len consecutive NaN."""
    isna = s.isna()
    run_id = isna.ne(isna.shift()).cumsum()
    run_len = isna.groupby(run_id).transform("sum")
    return isna & (run_len <= max_len)


def clean_series(df):
    out = to_regular(df).loc[WINDOW_START:WINDOW_END]
    for c, (low, high) in VALID_RANGES.items():
        bad = (out[c] < (-np.inf if low is None else low)) | (out[c] > (np.inf if high is None else high))
        out.loc[bad, c] = np.nan
    out["is_imputed"] = out[TARGET].isna().astype(int)
    for c in [TARGET, *NUM_EXOG]:
        filled = out[c].interpolate(method="time", limit_area="inside")
        out[c] = out[c].where(~short_gap_mask(out[c], MAX_INTERP_GAP), filled)
    for c in CAT_EXOG:
        out[c] = out[c].where(~short_gap_mask(out[c], MAX_INTERP_GAP), out[c].ffill())
    out.loc[out[TARGET].isna(), "is_imputed"] = 0   # still missing (long gap): not imputed, simply absent
    out["holiday"] = holiday_of(out.index)           # domain rule of 4.6
    return out


ts = clean_series(raw)
pd.Series({
    "steps in the study window": len(ts),
    "target interpolated (short gaps)": int(ts["is_imputed"].sum()),
    "target still missing (long gaps)": int(ts[TARGET].isna().sum()),
    "cells still missing (any column)": int(ts.isna().sum().sum()),
    "hours labelled as holiday": int(ts["holiday"].ne("None").sum()),
}, name="count").to_frame()

,count
steps in the study window,24096
target interpolated (short gaps),1012
target still missing (long gaps),0
cells still missing (any column),0
hours labelled as holiday,672


In [59]:
fig = go.Figure(go.Funnel(
    y=["raw rows", "unique hours (2012–2018)", "observed hours in the window",
       f"final hourly series<br>(+{int(ts['is_imputed'].sum()):,} interpolated hours)"],
    x=[len(raw), raw["timestamp"].nunique(), int((ts["is_imputed"] == 0).sum()), len(ts)],
    marker=dict(color=[MUTED, BLUE, BLUE, AQUA]), texttemplate="%{value:,}<br>%{percentInitial:.0%} of raw",
    connector=dict(fillcolor="#f0efec"), hovertemplate="%{y}: %{x:,} rows<extra></extra>",
))
show_fig(fig, f"Cleaning pipeline: {len(raw):,} raw rows become {len(ts):,} hours, one row per hour", height=380,
         margin=dict(l=230))

In [60]:
g = gap_table(is_missing[WINDOW_START:WINDOW_END]).iloc[0]
seg = ts.loc[g.start - pd.Timedelta(hours=36):g.end + pd.Timedelta(hours=36)]
imp = seg[seg["is_imputed"] == 1]
fig = go.Figure()
fig.add_vrect(x0=g.start - pd.Timedelta(minutes=30), x1=g.end + pd.Timedelta(minutes=30), fillcolor=ORANGE, opacity=0.08, line_width=0)
fig.add_trace(go.Scatter(x=seg.index, y=seg[TARGET], mode="lines", line=dict(color=BLUE, width=2), name="cleaned series",
                         hovertemplate="%{x|%d/%m %H:%M}: %{y:,.0f}<extra></extra>"))
fig.add_trace(go.Scatter(x=imp.index, y=imp[TARGET], mode="markers", name="interpolated hours",
                         marker=dict(color=ORANGE, size=9, line=dict(color="#ffffff", width=2)),
                         hovertemplate="%{x|%d/%m %H:%M}: %{y:,.0f} (interpolated)<extra></extra>"))
show_fig(fig, f"The longest gap of the window ({g.steps} hours) after time interpolation", height=400, yaxis_title="Vehicles/hour")

**Observation.** 24,096 hours, of which 1,012 (4.2%) were interpolated; no value is left missing. The longest gap (9 hours, an evening in February 2017) falls on the evening decline, so the straight line is a reasonable guess; the `is_imputed` flag still lets section 7 score models **on observed hours only**.

### 6.2 Feature engineering

Each row at time $t$ predicts $y_t$ using **only information available at $t - \text{HORIZON}$** (one hour earlier):

| Group | Features | Leakage guard |
|---|---|---|
| Lags | `lag_k` = $y_{t-k}$ for k in `LAGS` = 1, 2, 3, 24, 168 (5.6) | `LAGS` keeps k ≥ HORIZON |
| Rolling | mean / std of the target over the past 24 hours | computed on $y$ shifted by HORIZON |
| Calendar | hour, weekday, month as **sin/cos** pairs, `is_weekend` | known in advance |
| Holidays (domain, 5.8) | `is_holiday`, `is_major_holiday` | the holiday calendar is known in advance |
| Covariates | temperature, rain, snow, clouds, `n_weather`, `weather_main`, `weather_description` one hour earlier | shifted by HORIZON |

Sin/cos encoding puts 23h next to 0h and December next to January, which a plain integer cannot.

In [61]:
def cyclical(values, period):
    angle = 2 * np.pi * np.asarray(values) / period
    return np.sin(angle), np.cos(angle)


def make_features(ts):
    """Feature table for every time step; the target column is kept for training."""
    f = pd.DataFrame(index=ts.index)
    y = ts[TARGET]
    for k in LAGS:
        f[f"lag_{k}"] = y.shift(k)                   # shift(k): value k rows earlier (rows = time steps)
    past = y.shift(HORIZON)
    for w in ROLL_WINDOWS:
        f[f"roll_mean_{w}"] = past.rolling(w).mean()
        f[f"roll_std_{w}"] = past.rolling(w).std()
    idx = ts.index
    if INTRADAY:
        f["hour_sin"], f["hour_cos"] = cyclical(idx.hour, 24)
    f["dow_sin"], f["dow_cos"] = cyclical(idx.dayofweek, 7)
    f["month_sin"], f["month_cos"] = cyclical(idx.month - 1, 12)
    f["is_weekend"] = (idx.dayofweek >= 5).astype(int)
    holiday = holiday_of(idx)                        # domain features (5.8): calendar, known in advance
    f["is_holiday"] = holiday.ne("None").astype(int)
    f["is_major_holiday"] = holiday.isin(MAJOR_HOLIDAYS).astype(int)
    for c in [*NUM_EXOG, *CAT_EXOG]:
        f[f"{c}_lag{HORIZON}"] = ts[c].shift(HORIZON)
    f[TARGET] = y
    return f


features = make_features(ts)
CAT_FEATURES = [f"{c}_lag{HORIZON}" for c in CAT_EXOG]
NUM_FEATURES = [c for c in features.columns if c not in [TARGET, *CAT_FEATURES]]
FEATURE_COLS = NUM_FEATURES + CAT_FEATURES

model_df = features.dropna(subset=[*NUM_FEATURES, TARGET])  # drops warm-up rows (first max(LAGS) steps) and long gaps
print(f"{len(features):,} steps -> {len(model_df):,} usable rows ({len(features) - len(model_df):,} dropped)")
print(f"{len(NUM_FEATURES)} numeric + {len(CAT_FEATURES)} categorical features")
model_df.head()

24,096 steps -> 23,928 usable rows (168 dropped)
21 numeric + 2 categorical features


,lag_1,lag_2,lag_3,lag_24,lag_168,roll_mean_24,roll_std_24,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,is_weekend,is_holiday,is_major_holiday,temp_c_lag1,rain_1h_lag1,snow_1h_lag1,clouds_all_lag1,n_weather_lag1,weather_main_lag1,weather_description_lag1,traffic_volume
timestamp,,,,,,,,,,,,,,,,,,,,,,,,
2016-01-08 00:00:00,"1,241.000","2,075.500","2,910.000",558.000,"1,513.000","3,544.375","2,016.137",0.000,1.000,-0.434,-0.901,0.000,1.000,0,0,0,0.370,0.000,0.100,90.000,2.000,Snow,light snow,628.000
2016-01-08 01:00:00,628.000,"1,241.000","2,075.500",323.000,"1,550.000","3,547.292","2,011.675",0.259,0.966,-0.434,-0.901,0.000,1.000,0,0,0,0.200,0.000,0.060,75.000,4.000,Snow,snow,359.000
2016-01-08 02:00:00,359.000,628.000,"1,241.000",315.000,"1,134.500","3,548.792","2,009.178",0.500,0.866,-0.434,-0.901,0.000,1.000,0,0,0,0.130,0.000,0.170,90.000,3.000,Snow,light snow,380.000
2016-01-08 03:00:00,380.000,359.000,628.000,524.000,719.000,"3,551.500","2,004.668",0.707,0.707,-0.434,-0.901,0.000,1.000,0,0,0,0.100,0.000,0.125,90.000,2.500,Snow,light snow,401.000
2016-01-08 04:00:00,401.000,380.000,359.000,"1,081.000",533.000,"3,546.375","2,012.885",0.866,0.500,-0.434,-0.901,0.000,1.000,0,0,0,0.070,0.000,0.080,90.000,2.000,Snow,light snow,815.000


Why sin/cos? With the integer `hour`, 23h and 0h are 23 units apart although they are one hour apart. On the unit circle the distance is right:

In [62]:
hh = np.arange(24)
hs, hc = cyclical(hh, 24)
fig = go.Figure(go.Scatter(x=hs, y=hc, mode="markers+text", text=[f"{h}h" for h in hh], textposition="middle right",
                           textfont_color=INK_2,
                           marker=dict(size=14, color=pivot.mean().values, colorscale=SEQ_BLUE, showscale=True,
                                       colorbar=dict(title="Mean<br>vehicles"), line=dict(color=INK_2, width=1)),
                           hovertemplate="%{text}: sin=%{x:.2f}, cos=%{y:.2f}<extra></extra>"))
show_fig(fig, "Cyclical encoding: 23h sits next to 0h on the (hour_sin, hour_cos) circle", height=520,
         xaxis=dict(title="hour_sin", range=[-1.4, 1.5], scaleanchor="y"), yaxis=dict(title="hour_cos", range=[-1.3, 1.3]))

### 6.3 Chronological train / validation / test split

**No shuffling.** With a random split, `lag_1` of a test row would be the target of a training row, and the model would be scored on values it has effectively seen (leakage). Train is the past, validation the following period, test the most recent period:

| Set | Period | Role |
|---|---|---|
| Train | 08/01/2016 – 30/11/2017 | fit the models (the first week is used up by `lag_168`) |
| Validation | 01/12/2017 – 30/04/2018 | compare models and choose one |
| Test | 01/05/2018 – 30/09/2018 | score the chosen model once (section 8) |

In [63]:
t = model_df.index
train = model_df[t <= pd.Timestamp(TRAIN_END)]
val = model_df[(t > pd.Timestamp(TRAIN_END)) & (t <= pd.Timestamp(VAL_END))]
test = model_df[t > pd.Timestamp(VAL_END)]
assert len(train) and len(val) and len(test), "TRAIN_END / VAL_END leave an empty set"
assert train.index.max() < val.index.min() and val.index.max() < test.index.min()

split_table = pd.DataFrame({
    "from": [p.index.min() for p in (train, val, test)],
    "to": [p.index.max() for p in (train, val, test)],
    "rows": [len(p) for p in (train, val, test)],
    "share": [len(p) / len(model_df) for p in (train, val, test)],
    f"mean {TARGET}": [p[TARGET].mean() for p in (train, val, test)],
    "% interpolated": [ts.loc[p.index, "is_imputed"].mean() * 100 for p in (train, val, test)],
    "% snow hours": [(p[f"weather_main_lag{HORIZON}"] == "Snow").mean() * 100 for p in (train, val, test)],
}, index=["train", "validation", "test"])
display(split_table)

fig = go.Figure()
for name, part, color in [("train", train, BLUE), ("validation", val, ORANGE), ("test", test, AQUA)]:
    shown = part[TARGET].resample(OVERVIEW_FREQ).mean()
    fig.add_trace(go.Scatter(x=shown.index, y=shown.values, mode="lines", name=f"{name} ({len(part):,} rows)",
                             line=dict(color=color, width=1.6)))
for cut in (TRAIN_END, VAL_END):
    fig.add_vline(x=pd.Timestamp(cut), line_dash="dot", line_color=INK_2)
show_fig(fig, "Chronological split: train -> validation -> test, no overlap", height=420,
         yaxis_title="Vehicles/hour (daily mean)", hovermode="x unified")

,from,to,rows,share,mean traffic_volume,% interpolated,% snow hours
train,2016-01-08,2017-11-30 23:00:00,16632,0.695,"3,322.069",5.628,5.700
validation,2017-12-01,2018-04-30 23:00:00,3624,0.151,"3,240.124",0.469,17.632
test,2018-05-01,2018-09-30 23:00:00,3672,0.153,"3,358.080",0.163,0.000


In [64]:
fig = go.Figure()
for name, part, color in [("train", train, BLUE), ("validation", val, ORANGE), ("test", test, AQUA)]:
    counts, e = np.histogram(part[TARGET], bins=np.arange(0, 7600, 200))
    fig.add_trace(go.Scatter(x=(e[:-1] + e[1:]) / 2, y=counts / counts.sum() * 100, mode="lines", name=name,
                             line=dict(color=color, width=2.5, shape="spline"),
                             hovertemplate="%{x:,.0f}: %{y:.1f}% of hours<extra>" + name + "</extra>"))
show_fig(fig, "The target distribution has the same bimodal shape in the three sets", height=400,
         xaxis_title="Vehicles/hour", yaxis_title="% of hours (200-vehicle bins)")

**Observation.** Train holds 69.5% of the rows, validation and test about 15% each, with very similar means (≈ 3,240–3,360 vehicles/hour) and the same bimodal distribution. The sets differ by season: validation is winter–spring (**17.6% snow hours**), test is summer (no snow at all). The test score therefore says nothing about snowstorms, which must be stated in the report. Almost all interpolated hours are in train (5.6%); validation and test are ≥ 99.5% real data.

### 6.4 Scaling and encoding, fitted on train only

`StandardScaler` for numeric features (needed by Ridge; trees ignore it), `OneHotEncoder(handle_unknown="ignore")` for categorical covariates (a category first seen in test becomes all zeros instead of an error). Both live inside each model pipeline, so they are fitted on the training rows only, also inside `TimeSeriesSplit`.

In [65]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), NUM_FEATURES),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CAT_FEATURES),
])
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val, y_val = val[FEATURE_COLS], val[TARGET]
X_test, y_test = test[FEATURE_COLS], test[TARGET]

prep_check = clone(preprocessor).fit(X_train)
print(f"{len(FEATURE_COLS)} features -> {len(prep_check.get_feature_names_out())} model columns")
pd.DataFrame(prep_check.transform(X_train), columns=prep_check.get_feature_names_out()).describe().T[["mean", "std"]].head(12)

23 features -> 65 model columns


,mean,std
num__lag_1,0.000,1.000
num__lag_2,-0.000,1.000
num__lag_3,0.000,1.000
num__lag_24,0.000,1.000
num__lag_168,-0.000,1.000
num__roll_mean_24,0.000,1.000
num__roll_std_24,0.000,1.000
num__hour_sin,-0.000,1.000
num__hour_cos,-0.000,1.000
num__dow_sin,-0.000,1.000


Which features are linearly related to the target, and to each other? (training set only)

In [66]:
corr_cols = [TARGET, *[f"lag_{k}" for k in LAGS], "roll_mean_24", "hour_sin", "hour_cos", "is_weekend", "is_holiday",
             "is_major_holiday", *[f"{c}_lag{HORIZON}" for c in ["temp_c", "rain_1h", "snow_1h", "clouds_all"]]]
corr = train[corr_cols].corr()
z = corr.where(~np.triu(np.ones_like(corr, dtype=bool), k=1))
fig = go.Figure(go.Heatmap(z=z.values, x=corr.columns, y=corr.columns, zmin=-1, zmax=1, colorscale=DIVERGING,
                           text=z.round(2).astype(object).where(z.notna(), "").values, texttemplate="%{text}",
                           textfont=dict(size=10), xgap=2, ygap=2, colorbar=dict(title="r"),
                           hovertemplate="%{y} × %{x}: %{z:.2f}<extra></extra>"))
r_target = corr[TARGET].drop(TARGET)
show_fig(fig, f"Pearson correlation on train: lag_168 (r = {r_target['lag_168']:.2f}) and lag_1 ({r_target['lag_1']:.2f}) "
              "lead, the weather is weak", height=660, yaxis_autorange="reversed", xaxis_showgrid=False, yaxis_showgrid=False)

**Observation.** `lag_168` (r = 0.95), `lag_1` (0.91) and `lag_24` (0.87) are the strongest linear predictors, followed by `hour_cos` (−0.77); every weather variable has |r| ≤ 0.12. Pearson's r only measures *linear* links: the snow effect of 5.7 is non-linear and rare, so a small r does not make it useless for trees. Lags 1–3 are strongly correlated with each other (≥ 0.72), which is why the linear model is a **Ridge** (L2 penalty) rather than ordinary least squares.

<a id="s7"></a>
## 7. Modeling

### 7.1 Baselines and models on validation

| Model | Why |
|---|---|
| **Naive** | forecast = last known value ($y_{t-1}$) |
| **Seasonal naive** | forecast = value one cycle earlier ($y_{t-24}$ or $y_{t-168}$); often hard to beat on seasonal data |
| `Ridge` | linear combination of lags + calendar; L2 penalty keeps it stable with correlated lags |
| `RandomForestRegressor` | non-linear, averages many trees |
| `HistGradientBoostingRegressor` | boosted trees, strong on feature tables |
| **SARIMAX + Fourier** | the statistical reference: ARIMA errors on top of a weekly Fourier regression (sections 5.5–5.6) |

Metrics: **MAE** (average absolute error in vehicles/hour; the main metric), **RMSE** (punishes large errors), **MAPE** (relative, undefined near 0, so computed on non-zero targets only), **R²**, and **skill** = $1 - \text{MAE}_\text{model} / \text{MAE}_\text{naive}$ (> 0 means better than naive). With `EVAL_ON_OBSERVED_ONLY`, interpolated steps are excluded from scoring.

In [67]:
def score_mask(index):
    return (ts.loc[index, "is_imputed"] == 0).to_numpy() if EVAL_ON_OBSERVED_ONLY else np.ones(len(index), bool)


def regression_scores(y_true, y_pred):
    y_pred = np.asarray(y_pred, dtype=float)
    keep = score_mask(y_true.index) & ~np.isnan(y_pred)   # a baseline lag can fall into a long gap -> no forecast
    y_true, y_pred = y_true.to_numpy()[keep], y_pred[keep]
    err = y_true - y_pred
    nonzero = np.abs(y_true) > 1e-8
    return {"MAE": np.mean(np.abs(err)), "RMSE": np.sqrt(np.mean(err ** 2)),
            "MAPE %": np.mean(np.abs(err[nonzero] / y_true[nonzero])) * 100, "R2": r2_score(y_true, y_pred)}


BASELINES = {f"Naive (t-{HORIZON})": HORIZON,
             **{f"Seasonal naive (t-{p})": p for p in SEASONAL_PERIODS if p > HORIZON}}


def baseline_forecast(index, lag):
    return ts[TARGET].shift(lag).reindex(index).to_numpy()   # value `lag` steps before each target time


def make_model(estimator):
    return Pipeline([("prep", clone(preprocessor)), ("model", estimator)])


MODELS = {
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=SEED),
}

fitted, rows = {}, []
for name, lag in BASELINES.items():
    rows.append({"model": name, **regression_scores(y_val, baseline_forecast(y_val.index, lag))})
for name, estimator in MODELS.items():
    start = time.time()
    fitted[name] = make_model(estimator).fit(X_train, y_train)
    rows.append({"model": name, **regression_scores(y_val, fitted[name].predict(X_val))})
    print(f"done: {name} ({time.time() - start:.0f} s)")

results = pd.DataFrame(rows).set_index("model")
results["skill vs naive"] = 1 - results["MAE"] / results.loc[f"Naive (t-{HORIZON})", "MAE"]
results.sort_values("MAE")

done: Ridge (0 s)


done: Random Forest (5 s)


done: HistGradientBoosting (1 s)


,MAE,RMSE,MAPE %,R2,skill vs naive
model,,,,,
HistGradientBoosting,175.331,272.351,7.978,0.981,0.699
Random Forest,186.632,305.510,7.997,0.976,0.679
Ridge,318.845,447.361,17.209,0.949,0.452
Seasonal naive (t-168),449.097,781.492,18.125,0.843,0.228
Naive (t-1),581.654,802.258,27.347,0.835,0.000
Seasonal naive (t-24),629.087,"1,067.126",29.129,0.708,-0.082


**SARIMAX with Fourier terms (dynamic harmonic regression).** Section 5.6 ruled out a plain ARIMA (no seasonality) and a seasonal ARIMA with $m = 168$ (too heavy). The usual alternative is

$$y_t = c + \sum_{k=1}^{K}\left[a_k \sin\frac{2\pi k t}{168} + b_k \cos\frac{2\pi k t}{168}\right] + \beta^\top x_t + \eta_t, \qquad \eta_t \sim \text{AR}(3)$$

- the **Fourier terms** describe the weekly shape; harmonics $k = 7, 14, 21, \dots$ have a 24-hour period, so the daily cycle is included as well;
- $x_t$ holds the holiday flags and their interaction with the first three daily harmonics (a holiday changes the *shape* of the day, 5.8); all are known in advance;
- the **AR(3) errors** $\eta_t$ carry the short-term memory found by the PACF ($d = 0$, 5.6).

$K$ sets how detailed the weekly shape can be. It is chosen on validation among `FOURIER_K_GRID`. Forecasts are **one step ahead**: `apply(refit=False)` keeps the parameters fitted on train and filters the validation hours, so the forecast for hour $t$ uses observations up to $t-1$ only, exactly like the other models.

In [68]:
def sarimax_exog(index, K):
    """Weekly Fourier terms + holiday flags (+ holiday x daily harmonics) for every timestamp of `index`."""
    hours = ((index - pd.Timestamp("2016-01-04")) / pd.Timedelta(hours=1)).to_numpy()   # origin: a Monday 00:00
    X = {}
    for k in range(1, K + 1):
        X[f"sin_{k}"], X[f"cos_{k}"] = np.sin(2 * np.pi * k * hours / 168), np.cos(2 * np.pi * k * hours / 168)
    X = pd.DataFrame(X, index=index)
    holiday = holiday_of(index)
    X["is_holiday"] = holiday.ne("None").astype(float)
    X["is_major_holiday"] = holiday.isin(MAJOR_HOLIDAYS).astype(float)
    for k in (7, 14, 21):                          # 24h, 12h and 8h harmonics, switched on during holidays
        X[f"hol_sin_{k}"], X[f"hol_cos_{k}"] = X["is_holiday"] * X[f"sin_{k}"], X["is_holiday"] * X[f"cos_{k}"]
    return X


y_sx = ts[TARGET]                                  # cleaned series: no NaN left inside the window (6.1)
in_train = y_sx.index <= pd.Timestamp(TRAIN_END)
up_to_val = y_sx.index <= pd.Timestamp(VAL_END)
sarimax_rows, sarimax_fits = [], {}
for K in FOURIER_K_GRID:
    start = time.time()
    X_sx = sarimax_exog(y_sx.index, K)
    fit = SARIMAX(y_sx[in_train], exog=X_sx[in_train], order=ARIMA_ORDER, trend="c").fit(disp=False, maxiter=200)
    one_step = fit.apply(y_sx[up_to_val], exog=X_sx[up_to_val], refit=False).fittedvalues   # forecast of t uses data <= t-1
    sarimax_fits[K] = fit
    sarimax_rows.append({"K": K, "parameters": len(fit.params),
                         **regression_scores(y_val, one_step.reindex(y_val.index)), "seconds": time.time() - start})
sarimax_table = pd.DataFrame(sarimax_rows).set_index("K")
sarimax_table

,parameters,MAE,RMSE,MAPE %,R2,seconds
K,,,,,,
21,55,316.584,442.096,22.011,0.950,9.088
42,97,223.647,353.642,14.091,0.968,17.605
63,139,199.133,334.072,11.469,0.971,32.003


In [69]:
BEST_K = sarimax_table["MAE"].idxmin()
SARIMAX_NAME = f"SARIMAX{ARIMA_ORDER} + Fourier K={BEST_K}"
results.loc[SARIMAX_NAME] = pd.Series(regression_scores(y_val, sarimax_fits[BEST_K].apply(
    y_sx[up_to_val], exog=sarimax_exog(y_sx.index, BEST_K)[up_to_val], refit=False).fittedvalues.reindex(y_val.index)))
results["skill vs naive"] = 1 - results["MAE"] / results.loc[f"Naive (t-{HORIZON})", "MAE"]
print(f"AR coefficients: {sarimax_fits[BEST_K].params.filter(like='ar.').round(3).to_dict()}")
results.sort_values("MAE")

AR coefficients: {'ar.L1': 0.855, 'ar.L2': -0.027, 'ar.L3': 0.066}


,MAE,RMSE,MAPE %,R2,skill vs naive
model,,,,,
HistGradientBoosting,175.331,272.351,7.978,0.981,0.699
Random Forest,186.632,305.510,7.997,0.976,0.679
"SARIMAX(3, 0, 0) + Fourier K=63",199.133,334.072,11.469,0.971,0.658
Ridge,318.845,447.361,17.209,0.949,0.452
Seasonal naive (t-168),449.097,781.492,18.125,0.843,0.228
Naive (t-1),581.654,802.258,27.347,0.835,0.000
Seasonal naive (t-24),629.087,"1,067.126",29.129,0.708,-0.082


In [70]:
ordered = results.sort_values("MAE", ascending=False)
fig = go.Figure(go.Bar(x=ordered["MAE"], y=ordered.index, orientation="h",
                       marker_color=[MUTED if n in BASELINES else (ORANGE if n == SARIMAX_NAME else BLUE) for n in ordered.index],
                       text=[f"{m:,.0f}  ({s:+.0%} vs naive)" for m, s in zip(ordered["MAE"], ordered["skill vs naive"])],
                       textposition="outside", textfont_color=INK_2, hovertemplate="%{y}: MAE %{x:,.1f}<extra></extra>"))
show_fig(fig, f"Validation MAE (grey = baselines, orange = statistical model): best = {results['MAE'].idxmin()}",
         xaxis=dict(title="MAE (vehicles/hour)", range=[0, ordered["MAE"].max() * 1.35]), height=420, showlegend=False)

**Observation.**
- Every model beats the naive baselines by a wide margin. The best baseline is the **seasonal naive "same hour last week"** (MAE 449, +23% skill), as the ACF predicted; "same hour yesterday" is even worse than the plain naive forecast (629 vs 582) because it copies a Sunday into a Monday.
- The two tree models lead: **HistGradientBoosting MAE 175 (+70% skill)** and Random Forest 187. They can combine lags with the hour, the weekday and the holiday flags non-linearly (a holiday changes the *shape* of the day, snow matters only when it is heavy).
- **SARIMAX + Fourier is a strong third (MAE 199, +66%)**, far ahead of Ridge (319). More harmonics help a lot (K = 21: 317 → K = 42: 224 → K = 63: 199), because the weekday rush hours are sharp and need many harmonics; the AR(1) coefficient of 0.86 shows that the errors stay correlated from one hour to the next, which the AR part exploits. It uses only the calendar and the past of the series, no weather, and it stays fully interpretable, but every fit takes tens of seconds instead of one.
- Ridge is linear in `hour_sin` / `hour_cos`, so it cannot draw a two-peak day; it leans almost entirely on the lags.

### 7.2 Hyperparameter tuning with `TimeSeriesSplit`

Ordinary k-fold would train on the future and test on the past. `TimeSeriesSplit` creates folds where the training part always **precedes** the test part (expanding window), on the training set only.

In [71]:
TUNE_MODEL = "HistGradientBoosting"   # best ML model of 7.1
PARAM_GRID = {
    "model__learning_rate": [0.05, 0.1],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__max_iter": [300],
}
search = GridSearchCV(make_model(MODELS[TUNE_MODEL]), PARAM_GRID, scoring="neg_mean_absolute_error",
                      cv=TimeSeriesSplit(n_splits=3), n_jobs=-1)
search.fit(X_train, y_train)

tuned_name = f"{TUNE_MODEL} (tuned)"
fitted[tuned_name] = search.best_estimator_
results.loc[tuned_name] = pd.Series(regression_scores(y_val, search.best_estimator_.predict(X_val)))
results["skill vs naive"] = 1 - results["MAE"] / results.loc[f"Naive (t-{HORIZON})", "MAE"]
print("Best parameters:", search.best_params_, f"| CV MAE = {-search.best_score_:,.3f}")
results.sort_values("MAE")

Best parameters: {'model__learning_rate': 0.05, 'model__max_iter': 300, 'model__max_leaf_nodes': 31} | CV MAE = 189.665


,MAE,RMSE,MAPE %,R2,skill vs naive
model,,,,,
HistGradientBoosting (tuned),171.688,267.433,7.753,0.982,0.705
HistGradientBoosting,175.331,272.351,7.978,0.981,0.699
Random Forest,186.632,305.510,7.997,0.976,0.679
"SARIMAX(3, 0, 0) + Fourier K=63",199.133,334.072,11.469,0.971,0.658
Ridge,318.845,447.361,17.209,0.949,0.452
Seasonal naive (t-168),449.097,781.492,18.125,0.843,0.228
Naive (t-1),581.654,802.258,27.347,0.835,0.000
Seasonal naive (t-24),629.087,"1,067.126",29.129,0.708,-0.082


**Observation.** The best combination (learning rate 0.05, 31 leaves, 300 trees) reaches a cross-validated MAE of 190 on train and **172 on validation**, a 2% gain over the default settings (175): the default HistGradientBoosting was already close to its best.

### 7.3 Choose the final model

Best validation MAE among the scikit-learn pipelines, refitted on train + validation (the most recent data matters most for forecasting). The SARIMAX model stays in the comparison as the statistical reference.

In [72]:
ml_results = results.loc[[n for n in results.index if n in fitted]]
BEST_NAME = ml_results["MAE"].idxmin()
final_model = clone(fitted[BEST_NAME]).fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
print(f"Final model: {BEST_NAME} (validation MAE {results.loc[BEST_NAME, 'MAE']:,.3f}, "
      f"skill vs naive {results.loc[BEST_NAME, 'skill vs naive']:+.1%}); "
      f"overall best on validation: {results['MAE'].idxmin()}")

Final model: HistGradientBoosting (tuned) (validation MAE 171.688, skill vs naive +70.5%); overall best on validation: HistGradientBoosting (tuned)


<a id="s8"></a>
## 8. Evaluation on the test set

### 8.1 Test metrics

In [73]:
y_pred = pd.Series(final_model.predict(X_test), index=y_test.index)
rows = [{"model": name, **regression_scores(y_test, baseline_forecast(y_test.index, lag))} for name, lag in BASELINES.items()]
rows.append({"model": f"Final: {BEST_NAME}", **regression_scores(y_test, y_pred)})
test_scores = pd.DataFrame(rows).set_index("model")
test_scores["skill vs naive"] = 1 - test_scores["MAE"] / test_scores.loc[f"Naive (t-{HORIZON})", "MAE"]
test_scores

,MAE,RMSE,MAPE %,R2,skill vs naive
model,,,,,
Naive (t-1),582.834,810.077,26.036,0.828,0.000
Seasonal naive (t-24),511.980,981.742,21.992,0.748,0.122
Seasonal naive (t-168),257.240,535.070,10.406,0.925,0.559
Final: HistGradientBoosting (tuned),131.539,201.808,5.343,0.989,0.774


### 8.2 Forecast vs actual

In [74]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.12,
                    subplot_titles=("Whole test period (daily mean)", f"Last {ZOOM_STEPS} hours (two weeks)"))
for name, series, color in [("actual", y_test, BLUE), ("forecast", y_pred, ORANGE)]:
    coarse = series.resample(OVERVIEW_FREQ).mean()
    fig.add_trace(go.Scatter(x=coarse.index, y=coarse.values, mode="lines", name=name, legendgroup=name,
                             line=dict(color=color, width=1.6)), row=1, col=1)
    fine = series.iloc[-ZOOM_STEPS:]
    fig.add_trace(go.Scatter(x=fine.index, y=fine.values, mode="lines", name=name, legendgroup=name, showlegend=False,
                             line=dict(color=color, width=2)), row=2, col=1)
naive = pd.Series(baseline_forecast(y_test.index, HORIZON), index=y_test.index).iloc[-ZOOM_STEPS:]
fig.add_trace(go.Scatter(x=naive.index, y=naive.values, mode="lines", name=f"naive (t-{HORIZON})",
                         line=dict(color=MUTED, width=1.2, dash="dot")), row=2, col=1)
fig.update_yaxes(title_text="Vehicles/hour")
show_fig(fig, f"Test forecast vs actual: MAE {test_scores.iloc[-1]['MAE']:,.0f} vehicles/hour "
              f"({test_scores.iloc[-1]['skill vs naive']:+.0%} vs naive)", height=700, hovermode="x unified")

**Question: how does the forecast behave on a holiday?** The test period contains Memorial Day, Independence Day and Labor Day. Independence Day 2018 fell on a Wednesday:

In [75]:
HOLIDAY_ZOOM = slice("2018-07-02", "2018-07-06")
zoom_df = pd.DataFrame({"actual": y_test[HOLIDAY_ZOOM], "forecast": y_pred[HOLIDAY_ZOOM],
                        "seasonal naive (t-168)": baseline_forecast(y_test[HOLIDAY_ZOOM].index, 168)})
fig = go.Figure()
for col, color, dash in [("actual", BLUE, None), ("forecast", ORANGE, None), ("seasonal naive (t-168)", MUTED, "dot")]:
    fig.add_trace(go.Scatter(x=zoom_df.index, y=zoom_df[col], mode="lines", name=col, line=dict(color=color, width=2, dash=dash),
                             hovertemplate="%{y:,.0f}<extra>" + col + "</extra>"))
fig.add_vrect(x0=pd.Timestamp("2018-07-04"), x1=pd.Timestamp("2018-07-05"), fillcolor=AQUA, opacity=0.1, line_width=0,
              annotation_text="Independence Day", annotation_position="top left", annotation_font_color=AQUA)
hol_day = zoom_df.loc["2018-07-04"]
show_fig(fig, f"4 July 2018: model MAE {(hol_day['actual'] - hol_day['forecast']).abs().mean():,.0f} vs "
              f"{(hol_day['actual'] - hol_day['seasonal naive (t-168)']).abs().mean():,.0f} for the seasonal naive forecast",
         height=420, yaxis_title="Vehicles/hour", hovermode="x unified")

**Observation.** On the test period the forecast follows the actual series almost perfectly at the daily scale, and at the hourly scale it tracks every rush hour, while the naive forecast lags one hour behind. On **Independence Day** the seasonal naive forecast repeats an ordinary Wednesday and misses by 1,957 vehicles/hour on average; the model, which knows `is_holiday` / `is_major_holiday` and sees the low `lag_1` values in the morning, stays within 235 vehicles/hour. This is the comparison asked by question 5.8: the holiday features pay off exactly on the days the EDA singled out.

### 8.3 Where are the errors?

In [76]:
err = (y_test - y_pred)[score_mask(y_test.index)]
group = err.index.hour if INTRADAY else err.index.dayofweek
group_name = "hour" if INTRADAY else "weekday"
mae_by_group = err.abs().groupby(group).mean()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Residual distribution (actual − forecast)", f"MAE by {group_name}"))
fig.add_trace(hist_bar(err, bins=60), row=1, col=1)
fig.add_trace(go.Bar(x=mae_by_group.index, y=mae_by_group.values, marker_color=ORANGE,
                     hovertemplate=f"{group_name} %{{x}}: MAE %{{y:,.0f}}<extra></extra>"), row=1, col=2)
fig.update_xaxes(title_text="Error (vehicles/hour)", row=1, col=1)
fig.update_xaxes(title_text=group_name, row=1, col=2)
fig.update_yaxes(type="log", title_text="Hours (log scale)", row=1, col=1)
show_fig(fig, f"Residuals: mean {err.mean():+,.0f} (bias), largest MAE at {group_name} {mae_by_group.idxmax()}",
         height=420, showlegend=False)

### 8.4 Which features matter?

Permutation importance on test: shuffle one feature, measure the increase in MAE.

In [77]:
perm = permutation_importance(final_model, X_test, y_test, scoring="neg_mean_absolute_error", n_repeats=5,
                              random_state=SEED, n_jobs=-1)
importance = (pd.DataFrame({"MAE increase": perm.importances_mean, "std": perm.importances_std}, index=FEATURE_COLS)
                .sort_values("MAE increase").tail(15))
fig = go.Figure(go.Bar(x=importance["MAE increase"], y=importance.index, orientation="h", marker_color=BLUE,
                       error_x=dict(type="data", array=importance["std"], color=MUTED)))
show_fig(fig, f"Permutation importance on test: '{importance.index[-1]}' matters most",
         xaxis_title="Increase in MAE when the feature is shuffled (vehicles/hour)", height=max(380, 30 * len(importance)))

**Observation.** Errors are unbiased (mean −3 vehicles/hour) and symmetric with heavy tails. The MAE is lowest at night (< 50 at 1–4h) and highest at **23h (≈ 230)** and at the **7h morning peak (≈ 200)**: the steep morning rise and the late evening are where one hour of history says least about the next. Permutation importance confirms the EDA: shuffling `lag_1` raises the MAE by ≈ 1,300, then come the hour (`hour_cos`, ≈ 390) and `lag_168` (≈ 150), i.e. short-term memory, the daily cycle and the weekly cycle of sections 5.4–5.6. Weather and holiday features add almost nothing **on this test period**: it has no snow at all and only three holidays (6.3), so their value cannot show here even though they matter in winter (5.7).

### 8.5 Largest errors

Do the worst errors coincide with the outliers flagged in 4.5 (real but unpredictable events)?

In [78]:
worst = pd.DataFrame({"actual": y_test, "forecast": y_pred, "abs error": (y_test - y_pred).abs(),
                      "flagged outlier (4.5)": is_outlier.reindex(y_test.index, fill_value=False).astype(bool),
                      "holiday": holiday_of(y_test.index)})
share_flagged = worst.nlargest(50, "abs error")["flagged outlier (4.5)"].mean()
print(f"{share_flagged:.0%} of the 50 largest errors fall on steps flagged as outliers "
      f"(vs {worst['flagged outlier (4.5)'].mean():.1%} of all test steps)")
worst.nlargest(10, "abs error")

60% of the 50 largest errors fall on steps flagged as outliers (vs 3.1% of all test steps)


,actual,forecast,abs error,flagged outlier (4.5),holiday
timestamp,,,,,
2018-05-21 13:00:00,"2,040.000","4,798.171","2,758.171",True,None
2018-05-21 14:00:00,"4,224.000","2,094.517","2,129.483",True,None
2018-07-01 23:00:00,"3,078.000","1,397.210","1,680.790",True,None
2018-09-12 23:00:00,"2,842.000","1,213.115","1,628.885",True,None
2018-09-29 23:00:00,"3,856.000","2,297.827","1,558.173",True,None
2018-09-28 23:00:00,"3,544.000","2,009.012","1,534.988",True,None
2018-09-11 23:00:00,"2,682.000","1,224.689","1,457.311",True,None
2018-08-14 23:00:00,"2,772.000","1,427.707","1,344.293",True,None
2018-07-31 06:00:00,"4,240.000","5,557.386","1,317.386",True,None


**Observation.** 60% of the 50 largest errors fall on hours flagged as outliers in 4.5, although those are only 3.1% of the test hours: the worst misses are **unusual events**, not systematic errors. The largest one, 21 May 2018 at 13h, is a sudden drop to 2,040 vehicles (an incident) that the model only "sees" one hour later, so it is wrong twice (the drop, then the rebound). Several others are late evenings in September 2018 with 2,700–3,900 vehicles at 23h instead of the usual ≈ 1,300; the data does not say why (event traffic, a detour?). A one-hour-ahead model cannot anticipate either case without extra information such as incident or event feeds.

<a id="s9"></a>
## 9. Inference on new data

For a time series, "new data" means **the future**. After evaluation, the model is refitted on **all** usable history (train + validation + test), then forecasts the next `HORIZON` step after the last observation. The future row gets its features from `make_features`: lags ≥ HORIZON, rolling windows and covariates shifted by HORIZON all point to already observed hours; calendar and holiday features are known in advance.

In [79]:
deploy_model = clone(final_model).fit(model_df[FEATURE_COLS], model_df[TARGET])

last_time = ts.index.max()
future_index = pd.date_range(last_time, periods=HORIZON + 1, freq=FREQ)[1:]
extended = pd.concat([ts, pd.DataFrame(index=future_index)])     # future rows: target and covariates unknown (NaN)
future_X = make_features(extended).loc[future_index, FEATURE_COLS]
forecast = pd.Series(deploy_model.predict(future_X), index=future_index, name="forecast")

history = ts[TARGET].iloc[-2 * 24:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=history.index, y=history.values, mode="lines", name="history", line=dict(color=BLUE, width=2)))
fig.add_trace(go.Scatter(x=forecast.index, y=forecast.values, mode="lines+markers", name="forecast",
                         line=dict(color=ORANGE, width=2, dash="dash"), marker=dict(size=10)))
show_fig(fig, f"Forecast for {future_index[0]:%a %d/%m/%Y %H:%M}: {forecast.iloc[0]:,.0f} vehicles", height=420,
         yaxis_title="Vehicles/hour", hovermode="x unified")
display(future_X.T.rename(columns=lambda c: f"features for {c:%d/%m %H:%M}"))
forecast.to_frame()

,features for 01/10 00:00
lag_1,954.000
lag_2,"1,450.000"
lag_3,"2,159.000"
lag_24,"1,826.000"
lag_168,509.000
roll_mean_24,"2,504.292"
roll_std_24,"1,559.835"
hour_sin,0.000
hour_cos,1.000
dow_sin,0.000


,forecast
2018-10-01,570.541


**Observation.** The model refitted on all 23,928 usable hours forecasts **571 vehicles** for Monday 1 October 2018 at 00:00, continuing the Sunday-night decline (954 vehicles at 23h). The feature table above shows that every input of that row is known at forecast time: lags and covariates point to Sunday 23h or earlier, calendar and holiday flags are known in advance.

<a id="s10"></a>
## 10. Conclusion

### 10.1 Summary of results

**Test set** (May–September 2018, 3,672 hours, scored on observed hours only), used once:

| Model | MAE | RMSE | MAPE % | Skill vs naive |
|---|---|---|---|---|
| Naive (t-1) | 582.8 | 810.1 | 26.0 | 0 |
| Seasonal naive (t-168) | 257.2 | 535.1 | 10.4 | +55.9% |
| **HistGradientBoosting (tuned)** | **131.5** | **201.8** | **5.3** | **+77.4%** |

**Validation set** (December 2017 – April 2018), used to choose the model: tuned HistGradientBoosting 171.7, HistGradientBoosting 175.3, Random Forest 186.6, SARIMAX(3,0,0) + Fourier K = 63 199.1, Ridge 318.8, seasonal naive (t-168) 449.1, naive 581.7. The test MAE is lower than the validation MAE because the test period is summer without snow (6.3).

### 10.2 What the analysis answered

1. **Two nested cycles drive the series.** MSTL attributes 79% of the variance to the daily cycle and 15% to the weekly cycle, against 1% for the trend (Figures 18–19); the busiest hour carries 15 times the quietest, the busiest month only 12% more than the quietest (Figure 14).
2. **The most informative past value is the same hour last week** (ACF 0.94 at lag 168, Figure 21) together with the previous hour (0.91); these are also the top features of the final model (Figure 40).
3. **Holidays are not alike** (Figures 26–27): −28% on average, from −49% (New Year) to +5% (State Fair). With the holiday flags, the model misses Independence Day 2018 by 235 vehicles/hour instead of 1,957 for the seasonal naive forecast (Figure 38).
4. **Weather matters only when extreme**: snow −10.6% and heavy snow −43% against the same weekday-hour, all other weather within ±3% (Figures 23–24).
5. **Robust statistics are needed to find anomalies**: rolling median/MAD finds 14 of 14 road-closure hours, rolling mean/std 0 because of masking (Figure 28). The flagged hours account for 60% of the largest test errors (8.5).
6. **Tree models win, ARIMA is a solid reference**: tuned HistGradientBoosting cuts the naive MAE by 77% on test; SARIMAX with weekly Fourier terms and AR(3) errors reaches validation MAE 199, ahead of Ridge (319) and all baselines (Figure 36).

### 10.3 Limitations

- **Data**: one counting station and one direction (westbound); no information on incidents, road works or events, which cause the largest errors (8.5). Holiday labels come from the dataset only.
- **Method**: one-hour-ahead forecasts only; covariates are the weather observed one hour earlier, not a weather forecast. Interpolated hours (4.2%, mostly in train) are used for training.
- **Evaluation**: a single chronological split; the test period (summer 2018) has no snow and three holidays, so the winter behaviour is only measured on validation. SARIMAX was compared on validation only, as the test set is reserved for the chosen model.

### 10.4 Extensions

What was tried and compared in this notebook: weekly Fourier regression with ARIMA errors (three values of K) against tree models, and robust vs non-robust anomaly detection. Natural next steps: a longer horizon (e.g. 24 hours ahead, where `lag_1` is no longer available and the calendar and weekly lag must carry the forecast), and rolling-origin evaluation over several test windows, including a winter one.